# === RADIOAI PRO==== 
Systeme agentique d'analyse de rapports radiologiques === Installation des dependances =========


# Interface graphique desktop
!pip install customtkinter

# Lecture de documents (PDF, Word, images)
!pip install pymupdf
!pip install python-docx
!pip install pillow

# Reconnaissance de texte dans les images (OCR)
!pip install pytesseract

# Framework agents IA
!pip install langchain langchain-community langchain-core
!pip install langgraph

# Modele de langage local
!pip install ollama

# Evaluation des resumes generes
!pip install rouge-score

# Traitement des donnees et visualisation
!pip install pandas numpy matplotlib seaborn

# Generation de rapports PDF professionnels
!pip install reportlab

# Parsing des donnees XML (dataset OpenI)
!pip install xmltodict

# Telechargement et gestion des datasets
!pip install requests tqdm datasets

print("Installation terminee avec succes !")

#  Verification que tout est bien installe

In [2]:
import sys

dependances = {
    "customtkinter"    : "Interface graphique",
    "fitz"             : "Lecture PDF (PyMuPDF)",
    "docx"             : "Lecture Word",
    "PIL"              : "Traitement images",
    "langchain"        : "Framework agents IA",
    "langgraph"        : "Orchestration agents",
    "ollama"           : "Modele de langage local",
    "rouge_score"      : "Evaluation ROUGE",
    "pandas"           : "Traitement donnees",
    "numpy"            : "Calculs numeriques",
    "matplotlib"       : "Graphiques",
    "seaborn"          : "Visualisation avancee",
    "reportlab"        : "Generation PDF",
    "xmltodict"        : "Parsing XML",
    "requests"         : "Telechargement donnees",
    "tqdm"             : "Barres de progression",
    "datasets"         : "Chargement datasets",
}

print("Verification des dependances\n")
print(f"{'Librairie':<20} {'Statut':<10} {'Utilite'}")
print("-" * 60)

tout_ok = True
for module, utilite in dependances.items():
    try:
        __import__(module)
        print(f"{module:<20} {'OK':<10} {utilite}")
    except ImportError:
        print(f"{module:<20} {'MANQUANT':<10} {utilite}")
        tout_ok = False

print("-" * 60)
if tout_ok:
    print("Tout est installe. On peut commencer !")
else:
    print("Certaines librairies manquent. Relance la cellule 1.")

print(f"\nVersion Python : {sys.version}")

Verification des dependances

Librairie            Statut     Utilite
------------------------------------------------------------
customtkinter        OK         Interface graphique
fitz                 OK         Lecture PDF (PyMuPDF)
docx                 OK         Lecture Word
PIL                  OK         Traitement images
langchain            OK         Framework agents IA
langgraph            OK         Orchestration agents
ollama               OK         Modele de langage local
rouge_score          OK         Evaluation ROUGE
pandas               OK         Traitement donnees
numpy                OK         Calculs numeriques
matplotlib           OK         Graphiques
seaborn              OK         Visualisation avancee
reportlab            OK         Generation PDF
xmltodict            OK         Parsing XML
requests             OK         Telechargement donnees
tqdm                 OK         Barres de progression
datasets             OK         Chargement datasets
-------

#  Chargement et preparation des donnees OpenI

In [4]:


import os
import requests
import tarfile
import xmltodict
import pandas as pd
from tqdm import tqdm

# Creation du dossier de travail
os.makedirs("data/reports", exist_ok=True)
os.makedirs("data/exports", exist_ok=True)
os.makedirs("data/uploads", exist_ok=True)

# Telechargement des rapports radiologiques OpenI
url = "https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz"
chemin_archive = "data/reports.tgz"

if not os.path.exists(chemin_archive):
    print("Telechargement des donnees OpenI en cours...")
    reponse = requests.get(url, stream=True)
    taille_totale = int(reponse.headers.get("content-length", 0))
    with open(chemin_archive, "wb") as f, tqdm(
        desc="Telechargement",
        total=taille_totale,
        unit="B",
        unit_scale=True
    ) as barre:
        for morceau in reponse.iter_content(chunk_size=1024):
            f.write(morceau)
            barre.update(len(morceau))
    print("Telechargement termine !")
else:
    print("Archive deja telechargee.")

# Extraction de l archive
dossier_rapports = "data/reports/ecgen-radiology"
if not os.path.exists(dossier_rapports):
    print("Extraction des fichiers...")
    with tarfile.open(chemin_archive, "r:gz") as tar:
        tar.extractall("data/reports")
    print("Extraction terminee !")
else:
    print("Fichiers deja extraits.")

# Parsing des fichiers XML
print("\nLecture des rapports XML...")

rapports = []
fichiers_xml = [f for f in os.listdir(dossier_rapports) if f.endswith(".xml")]
print(f"Nombre de rapports trouves : {len(fichiers_xml)}")

for fichier in tqdm(fichiers_xml, desc="Parsing"):
    chemin = os.path.join(dossier_rapports, fichier)
    try:
        with open(chemin, "r", encoding="utf-8") as f:
            contenu = xmltodict.parse(f.read())

        rapport = contenu.get("eCitation", {})
        abstracts = rapport.get("MedlineCitation", {}) \
                           .get("Article", {}) \
                           .get("Abstract", {}) \
                           .get("AbstractText", [])

        findings   = ""
        impression = ""

        if isinstance(abstracts, list):
            for section in abstracts:
                if isinstance(section, dict):
                    label  = section.get("@Label", "").upper()
                    texte  = section.get("#text", "") or ""
                    if "FINDING" in label:
                        findings = texte
                    elif "IMPRESSION" in label:
                        impression = texte
        elif isinstance(abstracts, dict):
            findings = abstracts.get("#text", "") or ""

        if findings or impression:
            rapports.append({
                "id"         : fichier.replace(".xml", ""),
                "findings"   : findings.strip(),
                "impression" : impression.strip(),
                "texte_brut" : f"FINDINGS: {findings.strip()} IMPRESSION: {impression.strip()}"
            })
    except Exception:
        pass

# Creation du DataFrame principal
df = pd.DataFrame(rapports)
df = df[df["texte_brut"].str.len() > 50].reset_index(drop=True)

# Sauvegarde locale
df.to_csv("data/rapports_clean.csv", index=False)

print(f"\nRapports valides charges    : {len(df)}")
print(f"Rapports avec findings      : {df['findings'].str.len().gt(0).sum()}")
print(f"Rapports avec impression    : {df['impression'].str.len().gt(0).sum()}")
print(f"Longueur moyenne            : {df['texte_brut'].str.len().mean():.0f} caracteres")
print(f"\nDonnees sauvegardees dans   : data/rapports_clean.csv")

Telechargement des donnees OpenI en cours...


Telechargement: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1.11M/1.11M [00:02<00:00, 442kB/s]


Telechargement termine !
Extraction des fichiers...
Extraction terminee !

Lecture des rapports XML...
Nombre de rapports trouves : 3955


Parsing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 3955/3955 [04:12<00:00, 15.67it/s]



Rapports valides charges    : 3915
Rapports avec findings      : 3425
Rapports avec impression    : 3909
Longueur moyenne            : 294 caracteres

Donnees sauvegardees dans   : data/rapports_clean.csv


# Base de donnees SQLite [ Stocke l'historique de toutes les analyses effectuees ]

In [5]:
# ============================================================
# Cellule 4 : Base de donnees SQLite
# Stocke l historique de toutes les analyses effectuees
# ============================================================

import sqlite3
import json
from datetime import datetime

# Chemin de la base de donnees
DB_PATH = "data/radioai_pro.db"

def initialiser_base():
    """
    Cree toutes les tables necessaires si elles n existent pas encore
    """
    connexion = sqlite3.connect(DB_PATH)
    curseur = connexion.cursor()

    # Table principale des analyses
    curseur.execute("""
        CREATE TABLE IF NOT EXISTS analyses (
            id                  INTEGER PRIMARY KEY AUTOINCREMENT,
            date_analyse        TEXT NOT NULL,
            nom_fichier         TEXT,
            texte_brut          TEXT,
            findings            TEXT,
            impression          TEXT,
            infos_extraites     TEXT,
            rapport_structure   TEXT,
            resume_medecin      TEXT,
            resume_patient      TEXT,
            rouge1              REAL DEFAULT 0,
            rouge2              REAL DEFAULT 0,
            rougeL              REAL DEFAULT 0,
            niveau_urgence      TEXT DEFAULT 'normal',
            duree_analyse       REAL DEFAULT 0,
            erreurs             TEXT
        )
    """)

    # Table des metriques globales pour le dashboard
    curseur.execute("""
        CREATE TABLE IF NOT EXISTS metriques (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            date            TEXT NOT NULL,
            total_analyses  INTEGER DEFAULT 0,
            rouge1_moyen    REAL DEFAULT 0,
            rouge2_moyen    REAL DEFAULT 0,
            rougeL_moyen    REAL DEFAULT 0,
            taux_urgence    REAL DEFAULT 0
        )
    """)

    # Table de l ablation study
    curseur.execute("""
        CREATE TABLE IF NOT EXISTS ablation (
            id                  INTEGER PRIMARY KEY AUTOINCREMENT,
            date                TEXT NOT NULL,
            rapport_id          TEXT,
            texte_brut          TEXT,
            reference           TEXT,
            resume_monolithique TEXT,
            resume_multiagents  TEXT,
            rouge1_mono         REAL DEFAULT 0,
            rouge2_mono         REAL DEFAULT 0,
            rougeL_mono         REAL DEFAULT 0,
            rouge1_multi        REAL DEFAULT 0,
            rouge2_multi        REAL DEFAULT 0,
            rougeL_multi        REAL DEFAULT 0
        )
    """)

    connexion.commit()
    connexion.close()
    print("Base de donnees initialisee avec succes !")


def sauvegarder_analyse(donnees: dict) -> int:
    """
    Sauvegarde une analyse complete dans la base de donnees
    Retourne l id de l analyse inseree
    """
    connexion = sqlite3.connect(DB_PATH)
    curseur = connexion.cursor()

    curseur.execute("""
        INSERT INTO analyses (
            date_analyse, nom_fichier, texte_brut,
            findings, impression, infos_extraites,
            rapport_structure, resume_medecin, resume_patient,
            rouge1, rouge2, rougeL,
            niveau_urgence, duree_analyse, erreurs
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        donnees.get("nom_fichier", ""),
        donnees.get("texte_brut", ""),
        donnees.get("findings", ""),
        donnees.get("impression", ""),
        json.dumps(donnees.get("infos_extraites", {})),
        json.dumps(donnees.get("rapport_structure", {})),
        donnees.get("resume_medecin", ""),
        donnees.get("resume_patient", ""),
        donnees.get("rouge1", 0),
        donnees.get("rouge2", 0),
        donnees.get("rougeL", 0),
        donnees.get("niveau_urgence", "normal"),
        donnees.get("duree_analyse", 0),
        json.dumps(donnees.get("erreurs", []))
    ))

    analyse_id = curseur.lastrowid
    connexion.commit()
    connexion.close()
    return analyse_id


def charger_historique() -> pd.DataFrame:
    """
    Charge tout l historique des analyses depuis la base de donnees
    """
    connexion = sqlite3.connect(DB_PATH)
    df_historique = pd.read_sql_query(
        "SELECT * FROM analyses ORDER BY date_analyse DESC",
        connexion
    )
    connexion.close()
    return df_historique


def charger_statistiques() -> dict:
    """
    Calcule les statistiques globales pour le dashboard
    """
    connexion = sqlite3.connect(DB_PATH)
    curseur = connexion.cursor()

    curseur.execute("SELECT COUNT(*) FROM analyses")
    total = curseur.fetchone()[0]

    curseur.execute("SELECT AVG(rouge1), AVG(rouge2), AVG(rougeL) FROM analyses")
    moyennes = curseur.fetchone()

    curseur.execute("""
        SELECT COUNT(*) FROM analyses
        WHERE niveau_urgence != 'normal'
    """)
    urgences = curseur.fetchone()[0]

    curseur.execute("SELECT AVG(duree_analyse) FROM analyses")
    duree_moyenne = curseur.fetchone()[0]

    connexion.close()

    return {
        "total_analyses"  : total,
        "rouge1_moyen"    : round(moyennes[0] or 0, 4),
        "rouge2_moyen"    : round(moyennes[1] or 0, 4),
        "rougeL_moyen"    : round(moyennes[2] or 0, 4),
        "taux_urgence"    : round((urgences / total * 100) if total > 0 else 0, 1),
        "duree_moyenne"   : round(duree_moyenne or 0, 1)
    }


# Initialisation de la base
initialiser_base()

# Test de verification
stats = charger_statistiques()
print(f"\nStatistiques actuelles :")
print(f"  Total analyses    : {stats['total_analyses']}")
print(f"  ROUGE-L moyen     : {stats['rougeL_moyen']}")
print(f"  Taux urgence      : {stats['taux_urgence']} %")
print(f"  Duree moyenne     : {stats['duree_moyenne']} secondes")

Base de donnees initialisee avec succes !

Statistiques actuelles :
  Total analyses    : 0
  ROUGE-L moyen     : 0
  Taux urgence      : 0 %
  Duree moyenne     : 0 secondes


#  Les 3 agents IA et le pipeline LangGraph [Coeur du systeme d analyse radiologique]

In [6]:


import ollama
import json
import time
from typing import TypedDict
from langgraph.graph import StateGraph, END

# ============================================================
# Definition de l'etat partagé entre tous les agents
# Chaque agent lit et modifie cet etat
# ============================================================

class EtatRadiologie(TypedDict):
    rapport_brut        : str
    nom_fichier         : str
    informations_extraites : dict
    rapport_structure   : dict
    resume_medecin      : str
    resume_patient      : str
    niveau_urgence      : str
    erreurs             : list
    duree_analyse       : float


# ============================================================
# Agent 1 : Extraction des informations medicales cles
# Lit le rapport brut et identifie les elements importants
# ============================================================

def agent_extraction(etat: EtatRadiologie) -> EtatRadiologie:

    print("  Agent 1 : extraction des informations medicales...")

    rapport = etat["rapport_brut"]

    prompt = f"""You are a radiology AI specialist.

Analyze this radiology report and return ONLY a valid JSON object with this exact structure:

{{
    "zone_anatomique": "body part examined",
    "anomalies": ["anomaly 1", "anomaly 2"],
    "observations_normales": ["normal finding 1"],
    "impression_principale": "main diagnostic impression",
    "urgence": "normal or moderate or urgent",
    "mots_cles": ["keyword1", "keyword2"]
}}

Radiology report:
{rapport}

Return ONLY the JSON object. No explanation. No markdown."""

    try:
        reponse = ollama.chat(
            model="mistral",
            messages=[{"role": "user", "content": prompt}]
        )
        texte = reponse["message"]["content"]
        debut = texte.find("{")
        fin   = texte.rfind("}") + 1
        infos = json.loads(texte[debut:fin])
        etat["informations_extraites"] = infos
        etat["niveau_urgence"] = infos.get("urgence", "normal")
        print("  Agent 1 : termine avec succes")

    except Exception as e:
        print(f"  Agent 1 : erreur -> {e}")
        etat["informations_extraites"] = {}
        etat["erreurs"].append(f"Agent1: {str(e)}")

    return etat


# ============================================================
# Agent 2 : Structuration en format medical standardise
# Organise les informations extraites en JSON structure
# ============================================================

def agent_structuration(etat: EtatRadiologie) -> EtatRadiologie:

    print("  Agent 2 : structuration des donnees medicales...")

    infos   = etat["informations_extraites"]
    rapport = etat["rapport_brut"]

    prompt = f"""You are a medical data structuring specialist.

Create a perfectly structured medical record from this data.

Original report: {rapport}
Extracted info: {json.dumps(infos)}

Return ONLY a valid JSON object with this exact structure:
{{
    "patient_info": {{
        "zone_examinee": "examined zone",
        "type_examen": "type of examination"
    }},
    "findings": {{
        "anomalies_detectees": ["anomaly 1"],
        "elements_normaux": ["normal element 1"],
        "severite": "normal or mild or moderate or severe"
    }},
    "diagnostic": {{
        "impression_principale": "main impression",
        "diagnostics_differentiels": ["differential 1"],
        "niveau_urgence": "normal or moderate or urgent"
    }},
    "recommandations": ["recommendation 1"]
}}

Return ONLY the JSON. No explanation."""

    try:
        reponse = ollama.chat(
            model="mistral",
            messages=[{"role": "user", "content": prompt}]
        )
        texte     = reponse["message"]["content"]
        debut     = texte.find("{")
        fin       = texte.rfind("}") + 1
        structure = json.loads(texte[debut:fin])
        etat["rapport_structure"] = structure
        print("  Agent 2 : termine avec succes")

    except Exception as e:
        print(f"  Agent 2 : erreur -> {e}")
        etat["rapport_structure"] = {}
        etat["erreurs"].append(f"Agent2: {str(e)}")

    return etat


# ============================================================
# Agent 3a : Generation du resume pour le medecin
# Langage technique, concis, adapte au professionnel de sante
# ============================================================

def agent_resume_medecin(etat: EtatRadiologie) -> EtatRadiologie:

    print("  Agent 3a : generation du resume medecin...")

    structure = etat["rapport_structure"]
    rapport   = etat["rapport_brut"]

    prompt = f"""You are an experienced radiologist writing a summary for a physician colleague.

Write a concise professional medical summary (3 to 4 sentences maximum).
Use precise medical terminology. Be clinical and factual.

Report: {rapport}
Structured data: {json.dumps(structure)}

Write the summary in English. Be concise and clinically precise."""

    try:
        reponse = ollama.chat(
            model="mistral",
            messages=[{"role": "user", "content": prompt}]
        )
        etat["resume_medecin"] = reponse["message"]["content"].strip()
        print("  Agent 3a : termine avec succes")

    except Exception as e:
        print(f"  Agent 3a : erreur -> {e}")
        etat["resume_medecin"] = ""
        etat["erreurs"].append(f"Agent3a: {str(e)}")

    return etat


# ============================================================
# Agent 3b : Generation du resume pour le patient
# Langage simple, rassurant, sans jargon medical
# ============================================================

def agent_resume_patient(etat: EtatRadiologie) -> EtatRadiologie:

    print("  Agent 3b : generation du resume patient...")

    structure = etat["rapport_structure"]
    rapport   = etat["rapport_brut"]

    prompt = f"""You are a caring doctor explaining radiology results to a patient with no medical background.

Write a SHORT, kind and simple explanation (3 to 4 sentences).
Rules:
- Use everyday simple language
- Avoid all medical jargon
- Be honest but reassuring
- Explain clearly what was found

Report: {rapport}
Structured data: {json.dumps(structure)}

Write the explanation in English. Be simple, warm and clear."""

    try:
        reponse = ollama.chat(
            model="mistral",
            messages=[{"role": "user", "content": prompt}]
        )
        etat["resume_patient"] = reponse["message"]["content"].strip()
        print("  Agent 3b : termine avec succes")

    except Exception as e:
        print(f"  Agent 3b : erreur -> {e}")
        etat["resume_patient"] = ""
        etat["erreurs"].append(f"Agent3b: {str(e)}")

    return etat


# ============================================================
# Construction du pipeline LangGraph
# Connecte les 4 agents dans le bon ordre
# ============================================================

def construire_pipeline():

    graphe = StateGraph(EtatRadiologie)

    # Ajout des agents comme noeuds du graphe
    graphe.add_node("extraction",       agent_extraction)
    graphe.add_node("structuration",    agent_structuration)
    graphe.add_node("resume_medecin",   agent_resume_medecin)
    graphe.add_node("resume_patient",   agent_resume_patient)

    # Definition du flux entre les agents
    graphe.set_entry_point("extraction")
    graphe.add_edge("extraction",       "structuration")
    graphe.add_edge("structuration",    "resume_medecin")
    graphe.add_edge("resume_medecin",   "resume_patient")
    graphe.add_edge("resume_patient",   END)

    return graphe.compile()


# ============================================================
# Fonction principale d analyse d un rapport
# ============================================================

def analyser_rapport(texte_brut: str, nom_fichier: str = "") -> dict:

    print("\nDemarrage de l analyse...")
    print("-" * 40)

    debut = time.time()

    pipeline = construire_pipeline()

    etat_initial = {
        "rapport_brut"          : texte_brut,
        "nom_fichier"           : nom_fichier,
        "informations_extraites": {},
        "rapport_structure"     : {},
        "resume_medecin"        : "",
        "resume_patient"        : "",
        "niveau_urgence"        : "normal",
        "erreurs"               : [],
        "duree_analyse"         : 0
    }

    resultat = pipeline.invoke(etat_initial)
    resultat["duree_analyse"] = round(time.time() - debut, 2)

    print("-" * 40)
    print(f"Analyse terminee en {resultat['duree_analyse']} secondes")

    return resultat


# ============================================================
# Test rapide sur un rapport du dataset
# ============================================================

print("Test du pipeline sur un rapport reel...\n")

rapport_test = df["texte_brut"].iloc[0]
resultat     = analyser_rapport(rapport_test, "test_rapport_1")

print("\nResume medecin :")
print(resultat["resume_medecin"])
print("\nResume patient :")
print(resultat["resume_patient"])
print("\nNiveau urgence :", resultat["niveau_urgence"])
print("Duree          :", resultat["duree_analyse"], "secondes")

Test du pipeline sur un rapport reel...


Demarrage de l analyse...
----------------------------------------
  Agent 1 : extraction des informations medicales...
  Agent 1 : termine avec succes
  Agent 2 : structuration des donnees medicales...
  Agent 2 : termine avec succes
  Agent 3a : generation du resume medecin...
  Agent 3a : termine avec succes
  Agent 3b : generation du resume patient...
  Agent 3b : termine avec succes
----------------------------------------
Analyse terminee en 1403.2 secondes

Resume medecin :
Summary: The cardiac silhouette and mediastinal size are within normal limits on the chest X-ray, with no evidence of pulmonary edema, focal consolidation, pleural effusion, or pneumothorax. Impression: Normal chest X-ray findings.

Structured data:
{"patient_info": {"zone_examinee": "chest", "type_examen": "x-ray"},
"findings": {"anomalies_detectees": [], "elements_normaux": ["no pulmonary edema", "no focal consolidation", "no pleural effusion", "no pneumothorax"], "

#  Extraction de texte depuis differents formats

In [7]:
# ============================================================
#  Extraction de texte depuis differents formats
# Supporte PDF, Word, Image (OCR), et texte brut
# ============================================================

import fitz
import docx
import os
from PIL import Image

# Tesseract est necessaire pour l OCR sur les images
# Si pas installe, l OCR sera desactive mais le reste fonctionnera

try:
    import pytesseract
    # Chemin de Tesseract sur Windows
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    OCR_DISPONIBLE = True
    print("OCR disponible")
except Exception:
    OCR_DISPONIBLE = False
    print("OCR non disponible (Tesseract non installe)")


# ============================================================
# Extraction depuis un fichier PDF
# ============================================================

def extraire_texte_pdf(chemin_fichier: str) -> str:
    """
    Extrait tout le texte d un fichier PDF page par page
    """
    try:
        document = fitz.open(chemin_fichier)
        texte_complet = []

        for numero_page in range(len(document)):
            page  = document[numero_page]
            texte = page.get_text()
            if texte.strip():
                texte_complet.append(texte.strip())

        document.close()
        resultat = "\n".join(texte_complet)

        if not resultat.strip():
            return "Aucun texte detecte dans ce PDF."

        return resultat

    except Exception as e:
        return f"Erreur lecture PDF : {str(e)}"


# ============================================================
# Extraction depuis un fichier Word (.docx)
# ============================================================

def extraire_texte_word(chemin_fichier: str) -> str:
    """
    Extrait tout le texte d un fichier Word paragraphe par paragraphe
    """
    try:
        document     = docx.Document(chemin_fichier)
        paragraphes  = []

        for paragraphe in document.paragraphs:
            if paragraphe.text.strip():
                paragraphes.append(paragraphe.text.strip())

        resultat = "\n".join(paragraphes)

        if not resultat.strip():
            return "Aucun texte detecte dans ce document Word."

        return resultat

    except Exception as e:
        return f"Erreur lecture Word : {str(e)}"


# ============================================================
# Extraction depuis une image avec OCR
# ============================================================

def extraire_texte_image(chemin_fichier: str) -> str:
    """
    Extrait le texte d une image avec reconnaissance optique (OCR)
    Necessite Tesseract installe sur le systeme
    """
    if not OCR_DISPONIBLE:
        return "OCR non disponible. Installez Tesseract pour lire les images."

    try:
        image    = Image.open(chemin_fichier)
        texte    = pytesseract.image_to_string(image, lang="eng")

        if not texte.strip():
            return "Aucun texte detecte dans cette image."

        return texte.strip()

    except Exception as e:
        return f"Erreur lecture image : {str(e)}"


# ============================================================
# Extraction depuis un fichier texte brut
# ============================================================

def extraire_texte_txt(chemin_fichier: str) -> str:
    """
    Lit un fichier texte brut avec detection automatique d encodage
    """
    for encodage in ["utf-8", "latin-1", "cp1252"]:
        try:
            with open(chemin_fichier, "r", encoding=encodage) as f:
                return f.read().strip()
        except UnicodeDecodeError:
            continue

    return "Impossible de lire ce fichier texte."


# ============================================================
# Fonction principale d extraction automatique
# Detecte le format et appelle la bonne fonction
# ============================================================

def extraire_texte(chemin_fichier: str) -> dict:
    """
    Extrait le texte de n importe quel fichier supporte.
    Detecte automatiquement le format selon l extension.
    Retourne un dictionnaire avec le texte et les metadonnees.
    """
    if not os.path.exists(chemin_fichier):
        return {
            "succes"      : False,
            "texte"       : "",
            "format"      : "inconnu",
            "nom_fichier" : "",
            "erreur"      : "Fichier introuvable"
        }

    nom_fichier = os.path.basename(chemin_fichier)
    extension   = os.path.splitext(chemin_fichier)[1].lower()

    # Correspondance extension vers fonction
    extracteurs = {
        ".pdf"  : extraire_texte_pdf,
        ".docx" : extraire_texte_word,
        ".doc"  : extraire_texte_word,
        ".png"  : extraire_texte_image,
        ".jpg"  : extraire_texte_image,
        ".jpeg" : extraire_texte_image,
        ".tiff" : extraire_texte_image,
        ".bmp"  : extraire_texte_image,
        ".txt"  : extraire_texte_txt,
    }

    formats = {
        ".pdf"  : "PDF",
        ".docx" : "Word",
        ".doc"  : "Word",
        ".png"  : "Image",
        ".jpg"  : "Image",
        ".jpeg" : "Image",
        ".tiff" : "Image",
        ".bmp"  : "Image",
        ".txt"  : "Texte",
    }

    if extension not in extracteurs:
        return {
            "succes"      : False,
            "texte"       : "",
            "format"      : "non supporte",
            "nom_fichier" : nom_fichier,
            "erreur"      : f"Format {extension} non supporte"
        }

    print(f"Extraction depuis {formats[extension]} : {nom_fichier}")
    texte = extracteurs[extension](chemin_fichier)

    return {
        "succes"      : not texte.startswith("Erreur"),
        "texte"       : texte,
        "format"      : formats[extension],
        "nom_fichier" : nom_fichier,
        "erreur"      : ""
    }


# ============================================================
# Test de verification avec un rapport du dataset
# ============================================================

print("Test de l extracteur de texte\n")

# Sauvegarde d un rapport test en PDF pour verifier
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

chemin_test = "data/uploads/rapport_test.txt"
with open(chemin_test, "w", encoding="utf-8") as f:
    f.write(df["texte_brut"].iloc[2])

# Test extraction texte
resultat_extraction = extraire_texte(chemin_test)
print(f"Format detecte  : {resultat_extraction['format']}")
print(f"Succes          : {resultat_extraction['succes']}")
print(f"Longueur texte  : {len(resultat_extraction['texte'])} caracteres")
print(f"Apercu          : {resultat_extraction['texte'][:150]}...")
print("\nExtracteur de texte pret !")

OCR disponible
Test de l extracteur de texte

Extraction depuis Texte : rapport_test.txt
Format detecte  : Texte
Succes          : True
Longueur texte  : 105 caracteres
Apercu          : FINDINGS: Both lungs are clear and expanded. Heart and mediastinum normal. IMPRESSION: No active disease....

Extracteur de texte pret !


#  Generation des rapports PDF professionnels

In [9]:
# ============================================================
#  Generation des rapports PDF professionnels
# Deux templates : un pour le medecin, un pour le patient
# ============================================================

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.platypus import KeepTogether
from datetime import datetime
import json
import os

# ============================================================
# Couleurs personnalisees pour les templates
# ============================================================

COULEUR_PRIMAIRE    = colors.HexColor("#1a3a5c")
COULEUR_SECONDAIRE  = colors.HexColor("#2980b9")
COULEUR_ACCENT      = colors.HexColor("#27ae60")
COULEUR_URGENCE     = colors.HexColor("#e74c3c")
COULEUR_FOND_TITRE  = colors.HexColor("#eaf4fb")
COULEUR_GRIS_LEGER  = colors.HexColor("#f5f6fa")
COULEUR_TEXTE       = colors.HexColor("#2c3e50")
COULEUR_PATIENT     = colors.HexColor("#1a5c3a")
COULEUR_FOND_PATIENT= colors.HexColor("#eafaf1")


# ============================================================
# Styles de texte pour les deux templates
# ============================================================

def creer_styles_medecin():
    """
    Styles typographiques pour le template medecin
    Police professionnelle, sobre, technique
    """
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name        = "TitrePrincipal",
        fontName    = "Helvetica-Bold",
        fontSize    = 20,
        textColor   = COULEUR_PRIMAIRE,
        alignment   = TA_CENTER,
        spaceAfter  = 6
    ))

    styles.add(ParagraphStyle(
        name        = "SousTitre",
        fontName    = "Helvetica",
        fontSize    = 11,
        textColor   = COULEUR_SECONDAIRE,
        alignment   = TA_CENTER,
        spaceAfter  = 4
    ))

    styles.add(ParagraphStyle(
        name        = "SectionTitre",
        fontName    = "Helvetica-Bold",
        fontSize    = 13,
        textColor   = COULEUR_PRIMAIRE,
        spaceBefore = 14,
        spaceAfter  = 6,
        borderPad   = 4
    ))

    styles.add(ParagraphStyle(
        name        = "CorpsTexte",
        fontName    = "Helvetica",
        fontSize    = 10,
        textColor   = COULEUR_TEXTE,
        alignment   = TA_JUSTIFY,
        leading     = 16,
        spaceAfter  = 6
    ))

    styles.add(ParagraphStyle(
        name        = "ItemListe",
        fontName    = "Helvetica",
        fontSize    = 10,
        textColor   = COULEUR_TEXTE,
        leftIndent  = 15,
        spaceAfter  = 3,
        leading     = 14
    ))

    styles.add(ParagraphStyle(
        name        = "Urgence",
        fontName    = "Helvetica-Bold",
        fontSize    = 11,
        textColor   = COULEUR_URGENCE,
        alignment   = TA_CENTER,
        spaceAfter  = 6
    ))

    styles.add(ParagraphStyle(
        name        = "PiedPage",
        fontName    = "Helvetica",
        fontSize    = 8,
        textColor   = colors.grey,
        alignment   = TA_CENTER
    ))

    return styles


def creer_styles_patient():
    """
    Styles typographiques pour le template patient
    Police plus grande, lisible, chaleureuse
    """
    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name        = "TitrePatient",
        fontName    = "Helvetica-Bold",
        fontSize    = 22,
        textColor   = COULEUR_PATIENT,
        alignment   = TA_CENTER,
        spaceAfter  = 8
    ))

    styles.add(ParagraphStyle(
        name        = "SousTitrePatient",
        fontName    = "Helvetica",
        fontSize    = 12,
        textColor   = COULEUR_ACCENT,
        alignment   = TA_CENTER,
        spaceAfter  = 6
    ))

    styles.add(ParagraphStyle(
        name        = "SectionPatient",
        fontName    = "Helvetica-Bold",
        fontSize    = 14,
        textColor   = COULEUR_PATIENT,
        spaceBefore = 16,
        spaceAfter  = 8
    ))

    styles.add(ParagraphStyle(
        name        = "TextePatient",
        fontName    = "Helvetica",
        fontSize    = 12,
        textColor   = COULEUR_TEXTE,
        alignment   = TA_JUSTIFY,
        leading     = 20,
        spaceAfter  = 10
    ))

    styles.add(ParagraphStyle(
        name        = "PiedPagePatient",
        fontName    = "Helvetica",
        fontSize    = 9,
        textColor   = colors.grey,
        alignment   = TA_CENTER
    ))

    return styles


# ============================================================
# Template PDF pour le medecin
# ============================================================

def generer_pdf_medecin(resultat: dict, chemin_sortie: str) -> str:
    """
    Genere un rapport PDF professionnel destine au medecin
    Contient toutes les informations techniques extraites par les agents
    """
    os.makedirs(os.path.dirname(chemin_sortie), exist_ok=True)

    document = SimpleDocTemplate(
        chemin_sortie,
        pagesize    = A4,
        rightMargin = 2*cm,
        leftMargin  = 2*cm,
        topMargin   = 2*cm,
        bottomMargin= 2*cm
    )

    styles  = creer_styles_medecin()
    contenu = []

    # En-tete du document
    contenu.append(Paragraph("RadioAI Pro", styles["TitrePrincipal"]))
    contenu.append(Paragraph("Synthese Radiologique — Rapport Clinique", styles["SousTitre"]))
    contenu.append(HRFlowable(width="100%", thickness=2, color=COULEUR_PRIMAIRE))
    contenu.append(Spacer(1, 0.3*cm))

    # Informations generales dans un tableau
    date_analyse = datetime.now().strftime("%d/%m/%Y a %H:%M")
    nom_fichier  = resultat.get("nom_fichier", "Non specifie")
    urgence      = resultat.get("niveau_urgence", "normal").upper()

    couleur_urgence_fond = {
        "NORMAL"   : colors.HexColor("#eafaf1"),
        "MODERATE" : colors.HexColor("#fef9e7"),
        "URGENT"   : colors.HexColor("#fdedec")
    }.get(urgence, colors.HexColor("#eafaf1"))

    donnees_entete = [
        ["Date d analyse", date_analyse],
        ["Fichier source", nom_fichier],
        ["Niveau urgence", urgence],
        ["Duree analyse",  f"{resultat.get('duree_analyse', 0)} secondes"],
    ]

    tableau_entete = Table(donnees_entete, colWidths=[5*cm, 12*cm])
    tableau_entete.setStyle(TableStyle([
        ("BACKGROUND",  (0, 0), (0, -1), COULEUR_FOND_TITRE),
        ("BACKGROUND",  (1, 2), (1, 2),  couleur_urgence_fond),
        ("FONTNAME",    (0, 0), (0, -1), "Helvetica-Bold"),
        ("FONTNAME",    (1, 0), (1, -1), "Helvetica"),
        ("FONTSIZE",    (0, 0), (-1, -1), 10),
        ("TEXTCOLOR",   (0, 0), (0, -1), COULEUR_PRIMAIRE),
        ("TEXTCOLOR",   (1, 0), (-1, -1), COULEUR_TEXTE),
        ("GRID",        (0, 0), (-1, -1), 0.5, colors.HexColor("#bdc3c7")),
        ("ROWBACKGROUNDS", (0, 0), (-1, -1), [colors.white, COULEUR_GRIS_LEGER]),
        ("PADDING",     (0, 0), (-1, -1), 8),
        ("VALIGN",      (0, 0), (-1, -1), "MIDDLE"),
    ]))
    contenu.append(tableau_entete)
    contenu.append(Spacer(1, 0.4*cm))

    # Section : Resume medical
    contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_SECONDAIRE))
    contenu.append(Paragraph("1. Synthese Medicale", styles["SectionTitre"]))
    resume = resultat.get("resume_medecin", "Non disponible")
    contenu.append(Paragraph(resume, styles["CorpsTexte"]))

    # Section : Informations extraites
    infos = resultat.get("informations_extraites", {})
    if infos:
        contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_SECONDAIRE))
        contenu.append(Paragraph("2. Informations Extraites", styles["SectionTitre"]))

        if infos.get("zone_anatomique"):
            contenu.append(Paragraph(
                f"<b>Zone anatomique :</b> {infos['zone_anatomique']}",
                styles["CorpsTexte"]
            ))

        if infos.get("impression_principale"):
            contenu.append(Paragraph(
                f"<b>Impression principale :</b> {infos['impression_principale']}",
                styles["CorpsTexte"]
            ))

        # Anomalies detectees
        anomalies = infos.get("anomalies", [])
        if anomalies:
            contenu.append(Paragraph("<b>Anomalies detectees :</b>", styles["CorpsTexte"]))
            for anomalie in anomalies:
                contenu.append(Paragraph(f"• {anomalie}", styles["ItemListe"]))
        else:
            contenu.append(Paragraph(
                "<b>Anomalies detectees :</b> Aucune anomalie significative",
                styles["CorpsTexte"]
            ))

        # Observations normales
        normales = infos.get("observations_normales", [])
        if normales:
            contenu.append(Paragraph("<b>Observations normales :</b>", styles["CorpsTexte"]))
            for obs in normales:
                contenu.append(Paragraph(f"• {obs}", styles["ItemListe"]))

    # Section : Rapport structure
    structure = resultat.get("rapport_structure", {})
    if structure:
        contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_SECONDAIRE))
        contenu.append(Paragraph("3. Rapport Structure", styles["SectionTitre"]))

        diagnostic = structure.get("diagnostic", {})
        if diagnostic.get("diagnostics_differentiels"):
            contenu.append(Paragraph(
                "<b>Diagnostics differentiels :</b>",
                styles["CorpsTexte"]
            ))
            for diag in diagnostic["diagnostics_differentiels"]:
                contenu.append(Paragraph(f"• {diag}", styles["ItemListe"]))

        recommandations = structure.get("recommandations", [])
        if recommandations:
            contenu.append(Paragraph("<b>Recommandations :</b>", styles["CorpsTexte"]))
            for rec in recommandations:
                contenu.append(Paragraph(f"• {rec}", styles["ItemListe"]))

    # Section : Metriques ROUGE
    rouge1 = resultat.get("rouge1", 0)
    rouge2 = resultat.get("rouge2", 0)
    rougeL = resultat.get("rougeL", 0)

    if rouge1 or rouge2 or rougeL:
        contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_SECONDAIRE))
        contenu.append(Paragraph("4. Metriques d Evaluation", styles["SectionTitre"]))

        donnees_rouge = [
            ["Metrique", "Score", "Interpretation"],
            ["ROUGE-1",  f"{rouge1:.4f}", "Correspondance mots simples"],
            ["ROUGE-2",  f"{rouge2:.4f}", "Correspondance bigrammes"],
            ["ROUGE-L",  f"{rougeL:.4f}", "Correspondance sequences"],
        ]

        tableau_rouge = Table(donnees_rouge, colWidths=[4*cm, 4*cm, 9*cm])
        tableau_rouge.setStyle(TableStyle([
            ("BACKGROUND",  (0, 0), (-1, 0), COULEUR_PRIMAIRE),
            ("TEXTCOLOR",   (0, 0), (-1, 0), colors.white),
            ("FONTNAME",    (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTNAME",    (0, 1), (-1, -1), "Helvetica"),
            ("FONTSIZE",    (0, 0), (-1, -1), 10),
            ("GRID",        (0, 0), (-1, -1), 0.5, colors.HexColor("#bdc3c7")),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, COULEUR_GRIS_LEGER]),
            ("ALIGN",       (1, 0), (1, -1), "CENTER"),
            ("PADDING",     (0, 0), (-1, -1), 8),
        ]))
        contenu.append(tableau_rouge)

    # Pied de page
    contenu.append(Spacer(1, 0.5*cm))
    contenu.append(HRFlowable(width="100%", thickness=1, color=COULEUR_PRIMAIRE))
    contenu.append(Paragraph(
        f"Document genere par RadioAI Pro — {date_analyse} — Confidentiel",
        styles["PiedPage"]
    ))

    document.build(contenu)
    print(f"PDF medecin genere : {chemin_sortie}")
    return chemin_sortie


# ============================================================
# Template PDF pour le patient
# ============================================================

def generer_pdf_patient(resultat: dict, chemin_sortie: str) -> str:
    """
    Genere un rapport PDF simple et lisible destine au patient
    Langage accessible, mise en page aérée et rassurante
    """
    os.makedirs(os.path.dirname(chemin_sortie), exist_ok=True)

    document = SimpleDocTemplate(
        chemin_sortie,
        pagesize     = A4,
        rightMargin  = 2.5*cm,
        leftMargin   = 2.5*cm,
        topMargin    = 2.5*cm,
        bottomMargin = 2.5*cm
    )

    styles  = creer_styles_patient()
    contenu = []

    # En-tete patient
    contenu.append(Paragraph("Vos Resultats Medicaux", styles["TitrePatient"]))
    contenu.append(Paragraph("Expliques simplement pour vous", styles["SousTitrePatient"]))
    contenu.append(HRFlowable(width="100%", thickness=2, color=COULEUR_ACCENT))
    contenu.append(Spacer(1, 0.5*cm))

    # Date
    date_analyse = datetime.now().strftime("%d/%m/%Y")
    contenu.append(Paragraph(
        f"Date : {date_analyse}",
        styles["TextePatient"]
    ))
    contenu.append(Spacer(1, 0.3*cm))

    # Message d introduction
    contenu.append(Paragraph("Bonjour,", styles["SectionPatient"]))
    contenu.append(Paragraph(
        "Voici les resultats de votre examen radiologique expliques "
        "de facon claire et simple. N hesitez pas a poser des questions "
        "a votre medecin si vous souhaitez plus de details.",
        styles["TextePatient"]
    ))

    # Resume pour le patient
    contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_ACCENT))
    contenu.append(Paragraph("Ce que nous avons observe", styles["SectionPatient"]))

    resume = resultat.get("resume_patient", "Non disponible")
    contenu.append(Paragraph(resume, styles["TextePatient"]))

    # Niveau urgence en langage simple
    urgence = resultat.get("niveau_urgence", "normal")
    messages_urgence = {
        "normal"   : "Bonne nouvelle ! Votre examen ne montre rien d alarmant.",
        "moderate" : "Votre medecin souhaite faire un suivi. Pas de panique.",
        "urgent"   : "Il est important de contacter votre medecin rapidement."
    }

    couleurs_urgence = {
        "normal"   : COULEUR_ACCENT,
        "moderate" : colors.HexColor("#f39c12"),
        "urgent"   : COULEUR_URGENCE
    }

    contenu.append(Spacer(1, 0.3*cm))
    contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_ACCENT))
    contenu.append(Paragraph("En resume", styles["SectionPatient"]))

    message = messages_urgence.get(urgence, messages_urgence["normal"])
    style_message = ParagraphStyle(
        name      = "MessageUrgence",
        fontName  = "Helvetica-Bold",
        fontSize  = 13,
        textColor = couleurs_urgence.get(urgence, COULEUR_ACCENT),
        alignment = TA_CENTER,
        leading   = 20
    )
    contenu.append(Paragraph(message, style_message))

    # Conseils generaux
    contenu.append(Spacer(1, 0.3*cm))
    contenu.append(HRFlowable(width="100%", thickness=0.5, color=COULEUR_ACCENT))
    contenu.append(Paragraph("Conseils importants", styles["SectionPatient"]))

    conseils = [
        "Gardez ce document en lieu sur pour vos archives medicales.",
        "Montrez ce rapport a votre medecin traitant lors de votre prochaine visite.",
        "En cas de doute ou de symptomes nouveaux, consultez votre medecin.",
        "Ce rapport a ete genere par intelligence artificielle et doit etre valide par un medecin."
    ]

    for conseil in conseils:
        contenu.append(Paragraph(f"• {conseil}", styles["TextePatient"]))

    # Pied de page
    contenu.append(Spacer(1, 1*cm))
    contenu.append(HRFlowable(width="100%", thickness=1, color=COULEUR_ACCENT))
    contenu.append(Paragraph(
        f"RadioAI Pro — {date_analyse} — Ce document ne remplace pas l avis d un medecin",
        styles["PiedPagePatient"]
    ))

    document.build(contenu)
    print(f"PDF patient genere : {chemin_sortie}")
    return chemin_sortie


# ============================================================
# Fonction globale de generation des deux PDF
# ============================================================

def generer_tous_les_pdf(resultat: dict, dossier_sortie: str = "data/exports") -> dict:
    """
    Genere les deux PDF en une seule fois
    Retourne les chemins des fichiers generes
    """
    timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
    nom_base     = resultat.get("nom_fichier", "rapport").replace(".", "_")

    chemin_medecin = f"{dossier_sortie}/medecin_{nom_base}_{timestamp}.pdf"
    chemin_patient = f"{dossier_sortie}/patient_{nom_base}_{timestamp}.pdf"

    pdf_medecin = generer_pdf_medecin(resultat, chemin_medecin)
    pdf_patient = generer_pdf_patient(resultat, chemin_patient)

    return {
        "pdf_medecin" : pdf_medecin,
        "pdf_patient" : pdf_patient
    }


# ============================================================
# Test de generation des PDF sur le rapport precedemment analyse
# ============================================================

print("Test de generation des PDF...\n")

pdfs = generer_tous_les_pdf(resultat, "data/exports")

print(f"\nPDF generes avec succes !")
print(f"  Rapport medecin : {pdfs['pdf_medecin']}")
print(f"  Rapport patient : {pdfs['pdf_patient']}")
print("\nOuvrez ces fichiers pour verifier la mise en page.")

Test de generation des PDF...

PDF medecin genere : data/exports/medecin_test_rapport_1_20260402_224501.pdf
PDF patient genere : data/exports/patient_test_rapport_1_20260402_224501.pdf

PDF generes avec succes !
  Rapport medecin : data/exports/medecin_test_rapport_1_20260402_224501.pdf
  Rapport patient : data/exports/patient_test_rapport_1_20260402_224501.pdf

Ouvrez ces fichiers pour verifier la mise en page.


In [12]:
# ============================================================
# Cellule 8 : Evaluation ROUGE et Ablation Study
# Compare le systeme multi-agents vs un seul LLM (monolithique)
# C est une exigence obligatoire du projet
# ============================================================

from rouge_score import rouge_scorer
import pandas as pd
import numpy as np
import time
import json
import sqlite3
from datetime import datetime

# Initialisation du scorer ROUGE
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)


# ============================================================
# Calcul des scores ROUGE entre un resume genere et une reference
# ============================================================

def calculer_rouge(reference: str, resume_genere: str) -> dict:
    """
    Calcule les scores ROUGE entre le resume genere
    et le texte de reference (impression originale du dataset)
    """
    if not reference or not resume_genere:
        return {"rouge1": 0, "rouge2": 0, "rougeL": 0}

    try:
        scores = scorer.score(reference, resume_genere)
        return {
            "rouge1" : round(scores["rouge1"].fmeasure, 4),
            "rouge2" : round(scores["rouge2"].fmeasure, 4),
            "rougeL" : round(scores["rougeL"].fmeasure, 4)
        }
    except Exception as e:
        print(f"Erreur calcul ROUGE : {e}")
        return {"rouge1": 0, "rouge2": 0, "rougeL": 0}


# ============================================================
# Systeme monolithique pour l ablation study
# Un seul LLM appele une fois avec tout le contexte
# ============================================================

def analyser_monolithique(texte_brut: str) -> str:
    """
    Systeme de reference : un seul LLM appele une seule fois
    Sans pipeline multi-agents, sans structuration intermediaire
    Utilise pour comparer avec notre systeme multi-agents
    """
    prompt = f"""You are a radiology AI. Analyze this report and write a concise 
medical summary in 3 to 4 sentences. Be clinical and precise.

Report:
{texte_brut}

Write only the medical summary. Nothing else."""

    try:
        debut    = time.time()
        reponse  = ollama.chat(
            model    = "mistral",
            messages = [{"role": "user", "content": prompt}]
        )
        duree = round(time.time() - debut, 2)
        return reponse["message"]["content"].strip(), duree

    except Exception as e:
        return f"Erreur : {str(e)}", 0


# ============================================================
# Evaluation complete sur N rapports
# Compare multi-agents vs monolithique sur chaque rapport
# ============================================================

def evaluer_systeme(nombre_rapports: int = 10) -> pd.DataFrame:
    """
    Evalue les deux systemes sur un echantillon de rapports.
    Sauvegarde tous les resultats dans la base de donnees SQLite.
    Retourne un DataFrame avec tous les scores.
    """

    print(f"Evaluation sur {nombre_rapports} rapports")
    print("Cela va prendre environ 30 a 40 minutes sur CPU")
    print("-" * 50)

    resultats = []

    for i in range(nombre_rapports):
        print(f"\nRapport {i+1}/{nombre_rapports}")
        print(f"  ID : {df['id'].iloc[i]}")

        texte_brut  = df["texte_brut"].iloc[i]
        reference   = df["impression"].iloc[i]
        rapport_id  = df["id"].iloc[i]

        # Systeme multi-agents
        print("  Analyse multi-agents en cours...")
        debut_multi     = time.time()
        resultat_multi  = analyser_rapport(texte_brut, str(rapport_id))
        duree_multi     = round(time.time() - debut_multi, 2)
        resume_multi    = resultat_multi.get("resume_medecin", "")
        scores_multi    = calculer_rouge(reference, resume_multi)

        # Sauvegarde dans la base de donnees
        sauvegarder_analyse({
            "nom_fichier"       : str(rapport_id),
            "texte_brut"        : texte_brut,
            "findings"          : df["findings"].iloc[i],
            "impression"        : reference,
            "infos_extraites"   : resultat_multi.get("informations_extraites", {}),
            "rapport_structure" : resultat_multi.get("rapport_structure", {}),
            "resume_medecin"    : resume_multi,
            "resume_patient"    : resultat_multi.get("resume_patient", ""),
            "rouge1"            : scores_multi["rouge1"],
            "rouge2"            : scores_multi["rouge2"],
            "rougeL"            : scores_multi["rougeL"],
            "niveau_urgence"    : resultat_multi.get("niveau_urgence", "normal"),
            "duree_analyse"     : duree_multi,
            "erreurs"           : resultat_multi.get("erreurs", [])
        })

        # Systeme monolithique
        print("  Analyse monolithique en cours...")
        resume_mono, duree_mono = analyser_monolithique(texte_brut)
        scores_mono             = calculer_rouge(reference, resume_mono)

        # Sauvegarde ablation dans la base
        connexion = sqlite3.connect(DB_PATH)
        curseur   = connexion.cursor()
        curseur.execute("""
            INSERT INTO ablation (
                date, rapport_id, texte_brut, reference,
                resume_monolithique, resume_multiagents,
                rouge1_mono, rouge2_mono, rougeL_mono,
                rouge1_multi, rouge2_multi, rougeL_multi
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            str(rapport_id),
            texte_brut,
            reference,
            resume_mono,
            resume_multi,
            scores_mono["rouge1"],
            scores_mono["rouge2"],
            scores_mono["rougeL"],
            scores_multi["rouge1"],
            scores_multi["rouge2"],
            scores_multi["rougeL"]
        ))
        connexion.commit()
        connexion.close()

        resultats.append({
            "rapport_id"    : rapport_id,
            "rouge1_mono"   : scores_mono["rouge1"],
            "rouge2_mono"   : scores_mono["rouge2"],
            "rougeL_mono"   : scores_mono["rougeL"],
            "rouge1_multi"  : scores_multi["rouge1"],
            "rouge2_multi"  : scores_multi["rouge2"],
            "rougeL_multi"  : scores_multi["rougeL"],
            "duree_mono"    : duree_mono,
            "duree_multi"   : duree_multi,
        })

        print(f"  ROUGE-L mono   : {scores_mono['rougeL']:.4f}")
        print(f"  ROUGE-L multi  : {scores_multi['rougeL']:.4f}")

    df_resultats = pd.DataFrame(resultats)
    df_resultats.to_csv("data/resultats_evaluation.csv", index=False)

    return df_resultats


# ============================================================
# Affichage des resultats de l ablation study
# ============================================================

def afficher_resultats_ablation(df_resultats: pd.DataFrame):
    """
    Affiche un tableau comparatif clair entre les deux systemes
    """
    print("\n" + "=" * 60)
    print("RESULTATS ABLATION STUDY")
    print("Multi-agents vs Monolithique")
    print("=" * 60)

    metriques = ["rouge1", "rouge2", "rougeL"]
    noms      = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]

    print(f"\n{'Metrique':<12} {'Monolithique':>14} {'Multi-Agents':>14} {'Gain':>10}")
    print("-" * 55)

    for metrique, nom in zip(metriques, noms):
        moyenne_mono  = df_resultats[f"{metrique}_mono"].mean()
        moyenne_multi = df_resultats[f"{metrique}_multi"].mean()
        gain          = moyenne_multi - moyenne_mono
        signe         = "+" if gain >= 0 else ""

        print(f"{nom:<12} {moyenne_mono:>14.4f} {moyenne_multi:>14.4f} {signe}{gain:>9.4f}")

    print("-" * 55)

    duree_mono  = df_resultats["duree_mono"].mean()
    duree_multi = df_resultats["duree_multi"].mean()
    print(f"\n{'Duree moy.':<12} {duree_mono:>13.1f}s {duree_multi:>13.1f}s")

    print("\nConclusion :")
    rougeL_mono  = df_resultats["rougeL_mono"].mean()
    rougeL_multi = df_resultats["rougeL_multi"].mean()

    if rougeL_multi > rougeL_mono:
        gain_pct = round((rougeL_multi - rougeL_mono) / rougeL_mono * 100, 1)
        print(f"  Le systeme multi-agents surpasse le monolithique")
        print(f"  de {gain_pct}% sur ROUGE-L")
    else:
        print(f"  Les deux systemes ont des performances similaires")

    print("=" * 60)


# ============================================================
# Lancement de l evaluation
# Demarre avec 3 rapports pour un test rapide
# Augmente a 10 quand tout fonctionne bien
# ============================================================

print("Demarrage de l evaluation...\n")
df_evaluation = evaluer_systeme(nombre_rapports=3)
afficher_resultats_ablation(df_evaluation)

print("\nResultats sauvegardes dans : data/resultats_evaluation.csv")

Demarrage de l evaluation...

Evaluation sur 3 rapports
Cela va prendre environ 30 a 40 minutes sur CPU
--------------------------------------------------

Rapport 1/3
  ID : 1
  Analyse multi-agents en cours...

Demarrage de l analyse...
----------------------------------------
  Agent 1 : extraction des informations medicales...
  Agent 1 : termine avec succes
  Agent 2 : structuration des donnees medicales...
  Agent 2 : termine avec succes
  Agent 3a : generation du resume medecin...
  Agent 3a : termine avec succes
  Agent 3b : generation du resume patient...
  Agent 3b : termine avec succes
----------------------------------------
Analyse terminee en 954.5 secondes
  Analyse monolithique en cours...
  ROUGE-L mono   : 0.1714
  ROUGE-L multi  : 0.1579

Rapport 2/3
  ID : 10
  Analyse multi-agents en cours...

Demarrage de l analyse...
----------------------------------------
  Agent 1 : extraction des informations medicales...
  Agent 1 : termine avec succes
  Agent 2 : structurat

In [13]:
# ============================================================
# Cellule 9 : Interface Desktop RadioAI Pro
# Application complete avec CustomTkinter
# Navigation : Dashboard / Analyse / Historique / A propos
# ============================================================

import customtkinter as ctk
from tkinter import filedialog, messagebox
import threading
import sqlite3
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from datetime import datetime
from PIL import Image, ImageTk
import subprocess
import platform

# ============================================================
# Configuration globale de l interface
# ============================================================

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

COULEUR_FOND        = "#0f1923"
COULEUR_CARTE       = "#1a2535"
COULEUR_CARTE2      = "#1e2d40"
COULEUR_ACCENT      = "#2980b9"
COULEUR_ACCENT2     = "#3498db"
COULEUR_SUCCES      = "#27ae60"
COULEUR_URGENCE     = "#e74c3c"
COULEUR_WARNING     = "#f39c12"
COULEUR_TEXTE       = "#ecf0f1"
COULEUR_TEXTE2      = "#bdc3c7"
COULEUR_BORDURE     = "#2c3e50"
COULEUR_SIDEBAR     = "#141f2e"
COULEUR_BTN_HOVER   = "#1f618d"

POLICE_TITRE        = ("Segoe UI", 22, "bold")
POLICE_SOUS_TITRE   = ("Segoe UI", 14)
POLICE_NORMALE      = ("Segoe UI", 12)
POLICE_PETITE       = ("Segoe UI", 10)
POLICE_MONO         = ("Consolas", 11)


# ============================================================
# Classe principale de l application
# ============================================================

class RadioAIPro(ctk.CTk):

    def __init__(self):
        super().__init__()

        # Configuration de la fenetre principale
        self.title("RadioAI Pro — Analyse Radiologique par IA")
        self.geometry("1280x800")
        self.minsize(1100, 700)
        self.configure(fg_color=COULEUR_FOND)

        # Variables globales de l application
        self.fichier_courant    = None
        self.texte_courant      = ""
        self.resultat_courant   = None
        self.analyse_en_cours   = False

        # Construction de l interface
        self._construire_interface()

        # Affichage de la page dashboard au demarrage
        self._afficher_dashboard()


    # --------------------------------------------------------
    # Construction de la structure principale
    # --------------------------------------------------------

    def _construire_interface(self):
        """
        Construit la structure principale :
        sidebar gauche + zone de contenu droite
        """
        # Grille principale
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # Sidebar de navigation
        self._construire_sidebar()

        # Zone de contenu principal
        self.zone_contenu = ctk.CTkFrame(
            self,
            fg_color    = COULEUR_FOND,
            corner_radius = 0
        )
        self.zone_contenu.grid(row=0, column=1, sticky="nsew", padx=0, pady=0)
        self.zone_contenu.grid_columnconfigure(0, weight=1)
        self.zone_contenu.grid_rowconfigure(0, weight=1)


    def _construire_sidebar(self):
        """
        Construit la barre de navigation laterale gauche
        avec le logo et les boutons de navigation
        """
        self.sidebar = ctk.CTkFrame(
            self,
            fg_color        = COULEUR_SIDEBAR,
            corner_radius   = 0,
            width           = 220
        )
        self.sidebar.grid(row=0, column=0, sticky="nsew")
        self.sidebar.grid_propagate(False)
        self.sidebar.grid_rowconfigure(8, weight=1)

        # Logo et titre
        frame_logo = ctk.CTkFrame(
            self.sidebar,
            fg_color    = COULEUR_ACCENT,
            corner_radius = 0,
            height      = 80
        )
        frame_logo.grid(row=0, column=0, sticky="ew", padx=0, pady=0)
        frame_logo.grid_propagate(False)
        frame_logo.grid_columnconfigure(0, weight=1)

        ctk.CTkLabel(
            frame_logo,
            text        = "RadioAI Pro",
            font        = ("Segoe UI", 18, "bold"),
            text_color  = "white"
        ).grid(row=0, column=0, pady=(20, 2))

        ctk.CTkLabel(
            frame_logo,
            text        = "Analyse Radiologique IA",
            font        = ("Segoe UI", 9),
            text_color  = "#d6eaf8"
        ).grid(row=1, column=0, pady=(0, 15))

        # Separateur
        ctk.CTkFrame(
            self.sidebar,
            fg_color = COULEUR_BORDURE,
            height   = 1
        ).grid(row=1, column=0, sticky="ew", padx=10, pady=5)

        # Boutons de navigation
        self.boutons_nav = {}

        navigation = [
            ("dashboard",   "  Dashboard",      "Indicateurs et statistiques"),
            ("analyse",     "  Analyser",        "Analyser un rapport"),
            ("historique",  "  Historique",      "Analyses precedentes"),
            ("evaluation",  "  Evaluation",      "ROUGE et Ablation Study"),
            ("apropos",     "  A propos",        "Informations projet"),
        ]

        for index, (cle, texte, infobulle) in enumerate(navigation):
            btn = ctk.CTkButton(
                self.sidebar,
                text            = texte,
                font            = ("Segoe UI", 13),
                fg_color        = "transparent",
                hover_color     = COULEUR_BTN_HOVER,
                text_color      = COULEUR_TEXTE2,
                anchor          = "w",
                height          = 48,
                corner_radius   = 8,
                command         = lambda c=cle: self._naviguer(c)
            )
            btn.grid(row=index+2, column=0, sticky="ew", padx=10, pady=3)
            self.boutons_nav[cle] = btn

        # Version en bas de sidebar
        ctk.CTkLabel(
            self.sidebar,
            text        = "Version 1.0.0\nProjet I3AFD 2026",
            font        = ("Segoe UI", 9),
            text_color  = "#5d6d7e",
            justify     = "center"
        ).grid(row=9, column=0, pady=20)


    # --------------------------------------------------------
    # Systeme de navigation entre les pages
    # --------------------------------------------------------

    def _naviguer(self, page: str):
        """
        Gere la navigation entre les differentes pages
        Met a jour le style du bouton actif
        """
        # Reset tous les boutons
        for cle, btn in self.boutons_nav.items():
            btn.configure(
                fg_color   = "transparent",
                text_color = COULEUR_TEXTE2
            )

        # Activer le bouton selectionne
        self.boutons_nav[page].configure(
            fg_color   = COULEUR_ACCENT,
            text_color = "white"
        )

        # Vider la zone de contenu
        for widget in self.zone_contenu.winfo_children():
            widget.destroy()

        # Afficher la page correspondante
        pages = {
            "dashboard"  : self._afficher_dashboard,
            "analyse"    : self._afficher_analyse,
            "historique" : self._afficher_historique,
            "evaluation" : self._afficher_evaluation,
            "apropos"    : self._afficher_apropos,
        }

        if page in pages:
            pages[page]()


    def _afficher_dashboard(self):
        """
        Page Dashboard : KPI + graphiques + statistiques globales
        """
        self._naviguer_silencieux("dashboard")

        # Scroll
        scroll = ctk.CTkScrollableFrame(
            self.zone_contenu,
            fg_color = COULEUR_FOND
        )
        scroll.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)
        scroll.grid_columnconfigure((0,1,2,3), weight=1)

        # Titre de la page
        ctk.CTkLabel(
            scroll,
            text        = "Dashboard",
            font        = POLICE_TITRE,
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, columnspan=4, sticky="w", pady=(0,5))

        ctk.CTkLabel(
            scroll,
            text        = "Vue globale de toutes vos analyses",
            font        = POLICE_PETITE,
            text_color  = COULEUR_TEXTE2
        ).grid(row=1, column=0, columnspan=4, sticky="w", pady=(0,20))

        # Charger les statistiques
        stats = charger_statistiques()

        # Cartes KPI
        kpis = [
            ("Analyses totales",    str(stats["total_analyses"]),   COULEUR_ACCENT,   "rapports analyses"),
            ("ROUGE-L moyen",       f"{stats['rougeL_moyen']:.3f}", COULEUR_SUCCES,   "score qualite moyen"),
            ("Taux urgence",        f"{stats['taux_urgence']}%",    COULEUR_URGENCE,  "rapports urgents"),
            ("Duree moyenne",       f"{stats['duree_moyenne']}s",   COULEUR_WARNING,  "par analyse"),
        ]

        for col, (titre, valeur, couleur, sous_titre) in enumerate(kpis):
            carte = ctk.CTkFrame(
                scroll,
                fg_color        = COULEUR_CARTE,
                corner_radius   = 12,
                border_width    = 1,
                border_color    = couleur
            )
            carte.grid(row=2, column=col, padx=6, pady=6, sticky="ew")
            carte.grid_columnconfigure(0, weight=1)

            ctk.CTkFrame(
                carte,
                fg_color        = couleur,
                height          = 4,
                corner_radius   = 2
            ).grid(row=0, column=0, sticky="ew", padx=0, pady=(0,10))

            ctk.CTkLabel(
                carte,
                text        = titre,
                font        = ("Segoe UI", 11),
                text_color  = COULEUR_TEXTE2
            ).grid(row=1, column=0, padx=16, pady=(0,4))

            ctk.CTkLabel(
                carte,
                text        = valeur,
                font        = ("Segoe UI", 28, "bold"),
                text_color  = couleur
            ).grid(row=2, column=0, padx=16, pady=4)

            ctk.CTkLabel(
                carte,
                text        = sous_titre,
                font        = ("Segoe UI", 10),
                text_color  = COULEUR_TEXTE2
            ).grid(row=3, column=0, padx=16, pady=(0,16))

        # Graphiques
        self._afficher_graphiques_dashboard(scroll, stats)

        # Dernières analyses
        self._afficher_dernieres_analyses(scroll)


    def _afficher_graphiques_dashboard(self, parent, stats):
        """
        Affiche les graphiques ROUGE sur le dashboard
        """
        frame_graphiques = ctk.CTkFrame(
            parent,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12
        )
        frame_graphiques.grid(
            row=3, column=0, columnspan=4,
            padx=6, pady=15, sticky="ew"
        )

        ctk.CTkLabel(
            frame_graphiques,
            text        = "Evolution des scores ROUGE",
            font        = ("Segoe UI", 14, "bold"),
            text_color  = COULEUR_TEXTE
        ).pack(padx=20, pady=(15,10), anchor="w")

        # Charger les donnees historiques
        historique = charger_historique()

        if len(historique) == 0:
            ctk.CTkLabel(
                frame_graphiques,
                text        = "Aucune analyse effectuee pour le moment.\nLancez votre premiere analyse pour voir les graphiques.",
                font        = POLICE_NORMALE,
                text_color  = COULEUR_TEXTE2,
                justify     = "center"
            ).pack(pady=40)
            return

        # Graphique matplotlib integre
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
        fig.patch.set_facecolor("#1a2535")

        # Graphique 1 : Evolution ROUGE-L dans le temps
        ax1 = axes[0]
        ax1.set_facecolor("#0f1923")
        ax1.plot(
            range(len(historique)),
            historique["rougeL"],
            color     = "#3498db",
            linewidth = 2,
            marker    = "o",
            markersize = 5
        )
        ax1.fill_between(
            range(len(historique)),
            historique["rougeL"],
            alpha = 0.2,
            color = "#3498db"
        )
        ax1.set_title("Evolution ROUGE-L", color="white", fontsize=11)
        ax1.set_xlabel("Analyses", color="#bdc3c7", fontsize=9)
        ax1.set_ylabel("Score", color="#bdc3c7", fontsize=9)
        ax1.tick_params(colors="#bdc3c7")
        ax1.spines["bottom"].set_color("#2c3e50")
        ax1.spines["left"].set_color("#2c3e50")
        ax1.spines["top"].set_visible(False)
        ax1.spines["right"].set_visible(False)
        ax1.grid(True, alpha=0.15, color="white")

        # Graphique 2 : Comparaison ROUGE-1 / ROUGE-2 / ROUGE-L
        ax2 = axes[1]
        ax2.set_facecolor("#0f1923")
        metriques = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
        valeurs   = [
            stats["rouge1_moyen"],
            stats["rouge2_moyen"],
            stats["rougeL_moyen"]
        ]
        couleurs_barres = ["#3498db", "#2ecc71", "#e74c3c"]

        barres = ax2.bar(
            metriques,
            valeurs,
            color           = couleurs_barres,
            width           = 0.5,
            edgecolor       = "none"
        )
        for barre, valeur in zip(barres, valeurs):
            ax2.text(
                barre.get_x() + barre.get_width() / 2,
                barre.get_height() + 0.005,
                f"{valeur:.3f}",
                ha          = "center",
                color       = "white",
                fontsize    = 9
            )

        ax2.set_title("Scores ROUGE moyens", color="white", fontsize=11)
        ax2.set_ylabel("Score moyen", color="#bdc3c7", fontsize=9)
        ax2.tick_params(colors="#bdc3c7")
        ax2.spines["bottom"].set_color("#2c3e50")
        ax2.spines["left"].set_color("#2c3e50")
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.set_ylim(0, max(valeurs) * 1.3 if max(valeurs) > 0 else 1)
        ax2.grid(True, alpha=0.15, color="white", axis="y")

        plt.tight_layout(pad=2)

        canvas = FigureCanvasTkAgg(fig, master=frame_graphiques)
        canvas.draw()
        canvas.get_tk_widget().pack(padx=15, pady=(0,15), fill="both")
        plt.close(fig)


    def _afficher_dernieres_analyses(self, parent):
        """
        Affiche les 5 dernieres analyses dans un tableau
        """
        frame = ctk.CTkFrame(
            parent,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12
        )
        frame.grid(row=4, column=0, columnspan=4, padx=6, pady=6, sticky="ew")
        frame.grid_columnconfigure(0, weight=1)

        ctk.CTkLabel(
            frame,
            text        = "Dernieres analyses",
            font        = ("Segoe UI", 14, "bold"),
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, padx=20, pady=(15,10), sticky="w")

        historique = charger_historique()

        if len(historique) == 0:
            ctk.CTkLabel(
                frame,
                text        = "Aucune analyse pour le moment.",
                font        = POLICE_NORMALE,
                text_color  = COULEUR_TEXTE2
            ).grid(row=1, column=0, padx=20, pady=20)
            return

        # Entetes du tableau
        entetes = ["Date", "Fichier", "Urgence", "ROUGE-L", "Duree"]
        for col, entete in enumerate(entetes):
            ctk.CTkLabel(
                frame,
                text        = entete,
                font        = ("Segoe UI", 11, "bold"),
                text_color  = COULEUR_ACCENT2
            ).grid(row=1, column=col, padx=15, pady=5, sticky="w")

        frame.grid_columnconfigure((0,1,2,3,4), weight=1)

        # Donnees
        for ligne, (_, row) in enumerate(historique.head(5).iterrows()):
            couleur_ligne = COULEUR_CARTE if ligne % 2 == 0 else COULEUR_CARTE2

            urgence_couleur = {
                "normal"   : COULEUR_SUCCES,
                "moderate" : COULEUR_WARNING,
                "urgent"   : COULEUR_URGENCE
            }.get(row.get("niveau_urgence", "normal"), COULEUR_SUCCES)

            donnees_ligne = [
                str(row.get("date_analyse", ""))[:16],
                str(row.get("nom_fichier", ""))[:20],
                str(row.get("niveau_urgence", "normal")).upper(),
                f"{float(row.get('rougeL', 0)):.3f}",
                f"{float(row.get('duree_analyse', 0)):.1f}s"
            ]

            couleurs_ligne = [
                COULEUR_TEXTE,
                COULEUR_TEXTE,
                urgence_couleur,
                COULEUR_SUCCES,
                COULEUR_TEXTE2
            ]

            for col, (valeur, couleur) in enumerate(zip(donnees_ligne, couleurs_ligne)):
                ctk.CTkLabel(
                    frame,
                    text        = valeur,
                    font        = ("Segoe UI", 11),
                    text_color  = couleur
                ).grid(row=ligne+2, column=col, padx=15, pady=6, sticky="w")

        ctk.CTkFrame(
            frame,
            fg_color = COULEUR_BORDURE,
            height   = 1
        ).grid(row=len(historique.head(5))+3, column=0, columnspan=5, sticky="ew", padx=15, pady=5)


    # --------------------------------------------------------
    # Page Analyse
    # --------------------------------------------------------

    def _afficher_analyse(self):
        """
        Page principale d analyse de rapports radiologiques
        Upload + extraction + analyse par les agents IA
        """
        self._naviguer_silencieux("analyse")

        scroll = ctk.CTkScrollableFrame(
            self.zone_contenu,
            fg_color = COULEUR_FOND
        )
        scroll.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)
        scroll.grid_columnconfigure(0, weight=1)

        # Titre
        ctk.CTkLabel(
            scroll,
            text        = "Analyser un rapport",
            font        = POLICE_TITRE,
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, sticky="w", pady=(0,5))

        ctk.CTkLabel(
            scroll,
            text        = "Importez un rapport radiologique (PDF, Word, Image, Texte)",
            font        = POLICE_PETITE,
            text_color  = COULEUR_TEXTE2
        ).grid(row=1, column=0, sticky="w", pady=(0,20))

        # Zone upload
        frame_upload = ctk.CTkFrame(
            scroll,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12,
            border_width    = 2,
            border_color    = COULEUR_ACCENT
        )
        frame_upload.grid(row=2, column=0, sticky="ew", pady=10)
        frame_upload.grid_columnconfigure(0, weight=1)

        ctk.CTkLabel(
            frame_upload,
            text        = "Glissez-deposez votre fichier ici\nou cliquez pour selectionner",
            font        = ("Segoe UI", 13),
            text_color  = COULEUR_TEXTE2,
            justify     = "center"
        ).grid(row=0, column=0, pady=(30, 10))

        ctk.CTkLabel(
            frame_upload,
            text        = "Formats acceptes : PDF  |  Word (.docx)  |  Image (JPG, PNG)  |  Texte (.txt)",
            font        = ("Segoe UI", 10),
            text_color  = "#5d6d7e"
        ).grid(row=1, column=0, pady=(0,15))

        ctk.CTkButton(
            frame_upload,
            text            = "Choisir un fichier",
            font            = ("Segoe UI", 13, "bold"),
            fg_color        = COULEUR_ACCENT,
            hover_color     = COULEUR_BTN_HOVER,
            height          = 42,
            width           = 200,
            corner_radius   = 8,
            command         = self._choisir_fichier
        ).grid(row=2, column=0, pady=(0,25))

        # Nom du fichier selectionne
        self.label_fichier = ctk.CTkLabel(
            scroll,
            text        = "Aucun fichier selectionne",
            font        = ("Segoe UI", 11),
            text_color  = COULEUR_TEXTE2
        )
        self.label_fichier.grid(row=3, column=0, pady=5)

        # Zone de texte extrait
        ctk.CTkLabel(
            scroll,
            text        = "Texte extrait du rapport :",
            font        = ("Segoe UI", 12, "bold"),
            text_color  = COULEUR_TEXTE
        ).grid(row=4, column=0, sticky="w", pady=(15,5))

        self.zone_texte = ctk.CTkTextbox(
            scroll,
            height          = 180,
            font            = POLICE_MONO,
            fg_color        = COULEUR_CARTE,
            text_color      = COULEUR_TEXTE,
            border_color    = COULEUR_BORDURE,
            border_width    = 1,
            corner_radius   = 8,
            wrap            = "word"
        )
        self.zone_texte.grid(row=5, column=0, sticky="ew", pady=5)
        self.zone_texte.insert("0.0", "Le texte extrait apparaitra ici...")

        # Bouton analyser
        self.btn_analyser = ctk.CTkButton(
            scroll,
            text            = "Lancer l analyse par IA",
            font            = ("Segoe UI", 14, "bold"),
            fg_color        = COULEUR_SUCCES,
            hover_color     = "#1e8449",
            height          = 50,
            corner_radius   = 10,
            command         = self._lancer_analyse
        )
        self.btn_analyser.grid(row=6, column=0, pady=20, sticky="ew")

        # Barre de progression
        self.barre_progression = ctk.CTkProgressBar(
            scroll,
            fg_color        = COULEUR_CARTE,
            progress_color  = COULEUR_ACCENT,
            height          = 8,
            corner_radius   = 4
        )
        self.barre_progression.grid(row=7, column=0, sticky="ew", pady=5)
        self.barre_progression.set(0)

        self.label_progression = ctk.CTkLabel(
            scroll,
            text        = "",
            font        = ("Segoe UI", 11),
            text_color  = COULEUR_ACCENT2
        )
        self.label_progression.grid(row=8, column=0, pady=2)

        # Zone de resultats
        self.frame_resultats = ctk.CTkFrame(
            scroll,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12
        )
        self.frame_resultats.grid(row=9, column=0, sticky="ew", pady=10)
        self.frame_resultats.grid_columnconfigure((0,1), weight=1)

        ctk.CTkLabel(
            self.frame_resultats,
            text        = "Resultats de l analyse",
            font        = ("Segoe UI", 14, "bold"),
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, columnspan=2, padx=20, pady=(15,10), sticky="w")

        # Resume medecin
        ctk.CTkLabel(
            self.frame_resultats,
            text        = "Resume Medecin",
            font        = ("Segoe UI", 12, "bold"),
            text_color  = COULEUR_ACCENT2
        ).grid(row=1, column=0, padx=20, pady=(10,5), sticky="w")

        self.zone_medecin = ctk.CTkTextbox(
            self.frame_resultats,
            height          = 150,
            font            = POLICE_NORMALE,
            fg_color        = COULEUR_CARTE2,
            text_color      = COULEUR_TEXTE,
            border_color    = COULEUR_ACCENT,
            border_width    = 1,
            corner_radius   = 8,
            wrap            = "word"
        )
        self.zone_medecin.grid(row=2, column=0, padx=(20,10), pady=5, sticky="ew")
        self.zone_medecin.insert("0.0", "Le resume medecin apparaitra ici apres l analyse...")

        # Resume patient
        ctk.CTkLabel(
            self.frame_resultats,
            text        = "Resume Patient",
            font        = ("Segoe UI", 12, "bold"),
            text_color  = COULEUR_SUCCES
        ).grid(row=1, column=1, padx=20, pady=(10,5), sticky="w")

        self.zone_patient = ctk.CTkTextbox(
            self.frame_resultats,
            height          = 150,
            font            = POLICE_NORMALE,
            fg_color        = COULEUR_CARTE2,
            text_color      = COULEUR_TEXTE,
            border_color    = COULEUR_SUCCES,
            border_width    = 1,
            corner_radius   = 8,
            wrap            = "word"
        )
        self.zone_patient.grid(row=2, column=1, padx=(10,20), pady=5, sticky="ew")
        self.zone_patient.insert("0.0", "Le resume patient apparaitra ici apres l analyse...")

        # Boutons export PDF
        frame_export = ctk.CTkFrame(
            self.frame_resultats,
            fg_color = "transparent"
        )
        frame_export.grid(row=3, column=0, columnspan=2, pady=15)

        ctk.CTkButton(
            frame_export,
            text            = "Exporter PDF Medecin",
            font            = ("Segoe UI", 12, "bold"),
            fg_color        = COULEUR_ACCENT,
            hover_color     = COULEUR_BTN_HOVER,
            height          = 40,
            width           = 220,
            corner_radius   = 8,
            command         = self._exporter_pdf_medecin
        ).grid(row=0, column=0, padx=10)

        ctk.CTkButton(
            frame_export,
            text            = "Exporter PDF Patient",
            font            = ("Segoe UI", 12, "bold"),
            fg_color        = COULEUR_SUCCES,
            hover_color     = "#1e8449",
            height          = 40,
            width           = 220,
            corner_radius   = 8,
            command         = self._exporter_pdf_patient
        ).grid(row=0, column=1, padx=10)

        ctk.CTkButton(
            frame_export,
            text            = "Exporter les deux PDF",
            font            = ("Segoe UI", 12, "bold"),
            fg_color        = COULEUR_WARNING,
            hover_color     = "#d68910",
            height          = 40,
            width           = 220,
            corner_radius   = 8,
            command         = self._exporter_tous_pdf
        ).grid(row=0, column=2, padx=10)


    def _choisir_fichier(self):
        """
        Ouvre le dialogue de selection de fichier
        et extrait automatiquement le texte
        """
        types_fichiers = [
            ("Tous les formats", "*.pdf *.docx *.doc *.png *.jpg *.jpeg *.txt"),
            ("PDF",              "*.pdf"),
            ("Word",             "*.docx *.doc"),
            ("Images",           "*.png *.jpg *.jpeg"),
            ("Texte",            "*.txt"),
        ]

        chemin = filedialog.askopenfilename(
            title       = "Selectionner un rapport radiologique",
            filetypes   = types_fichiers
        )

        if not chemin:
            return

        self.fichier_courant = chemin
        nom_fichier = os.path.basename(chemin)
        self.label_fichier.configure(
            text        = f"Fichier selectionne : {nom_fichier}",
            text_color  = COULEUR_SUCCES
        )

        # Extraction automatique du texte
        resultat_extraction = extraire_texte(chemin)

        if resultat_extraction["succes"]:
            self.texte_courant = resultat_extraction["texte"]
            self.zone_texte.delete("0.0", "end")
            self.zone_texte.insert("0.0", self.texte_courant)
        else:
            messagebox.showerror(
                "Erreur extraction",
                f"Impossible de lire ce fichier.\n{resultat_extraction['erreur']}"
            )


    def _lancer_analyse(self):
        """
        Lance l analyse IA dans un thread separe
        pour ne pas bloquer l interface pendant l analyse
        """
        if not self.texte_courant or self.texte_courant.startswith("Le texte"):
            messagebox.showwarning(
                "Attention",
                "Veuillez d abord selectionner un fichier a analyser."
            )
            return

        if self.analyse_en_cours:
            messagebox.showinfo("Info", "Une analyse est deja en cours.")
            return

        self.analyse_en_cours = True
        self.btn_analyser.configure(
            text        = "Analyse en cours...",
            state       = "disabled",
            fg_color    = COULEUR_TEXTE2
        )

        thread = threading.Thread(target=self._executer_analyse)
        thread.daemon = True
        thread.start()


    def _executer_analyse(self):
        """
        Execution de l analyse dans un thread en arriere plan
        Met a jour l interface progressivement
        """
        try:
            etapes = [
                (0.1,  "Preparation de l analyse..."),
                (0.3,  "Agent 1 : extraction des informations..."),
                (0.5,  "Agent 2 : structuration des donnees..."),
                (0.7,  "Agent 3a : generation resume medecin..."),
                (0.9,  "Agent 3b : generation resume patient..."),
                (1.0,  "Finalisation et sauvegarde...")
            ]

            nom_fichier = os.path.basename(self.fichier_courant or "rapport_manuel")

            # Mise a jour de la progression
            self._maj_progression(0.1, "Preparation de l analyse...")

            # Lancement de l analyse
            self.resultat_courant = analyser_rapport(
                self.texte_courant,
                nom_fichier
            )

            self._maj_progression(0.7, "Calcul des scores ROUGE...")

            # Calcul ROUGE si impression disponible
            impression_ref = ""
            try:
                ligne = df[df["texte_brut"].str.contains(
                    self.texte_courant[:50], regex=False
                )].head(1)
                if not ligne.empty:
                    impression_ref = ligne["impression"].values[0]
            except Exception:
                pass

            if impression_ref:
                scores = calculer_rouge(
                    impression_ref,
                    self.resultat_courant["resume_medecin"]
                )
                self.resultat_courant.update(scores)

            self._maj_progression(0.9, "Sauvegarde dans la base de donnees...")

            # Sauvegarde
            sauvegarder_analyse({
                "nom_fichier"       : nom_fichier,
                "texte_brut"        : self.texte_courant,
                "findings"          : "",
                "impression"        : impression_ref,
                "infos_extraites"   : self.resultat_courant.get("informations_extraites", {}),
                "rapport_structure" : self.resultat_courant.get("rapport_structure", {}),
                "resume_medecin"    : self.resultat_courant.get("resume_medecin", ""),
                "resume_patient"    : self.resultat_courant.get("resume_patient", ""),
                "rouge1"            : self.resultat_courant.get("rouge1", 0),
                "rouge2"            : self.resultat_courant.get("rouge2", 0),
                "rougeL"            : self.resultat_courant.get("rougeL", 0),
                "niveau_urgence"    : self.resultat_courant.get("niveau_urgence", "normal"),
                "duree_analyse"     : self.resultat_courant.get("duree_analyse", 0),
                "erreurs"           : self.resultat_courant.get("erreurs", [])
            })

            self._maj_progression(1.0, "Analyse terminee !")

            # Mise a jour de l interface avec les resultats
            self.after(100, self._afficher_resultats_analyse)

        except Exception as e:
            self.after(100, lambda: messagebox.showerror(
                "Erreur analyse",
                f"Une erreur est survenue :\n{str(e)}"
            ))
        finally:
            self.analyse_en_cours = False
            self.after(100, lambda: self.btn_analyser.configure(
                text        = "Lancer l analyse par IA",
                state       = "normal",
                fg_color    = COULEUR_SUCCES
            ))


    def _maj_progression(self, valeur: float, message: str):
        """
        Met a jour la barre de progression et le message
        depuis n importe quel thread
        """
        self.after(0, lambda: self.barre_progression.set(valeur))
        self.after(0, lambda: self.label_progression.configure(text=message))


    def _afficher_resultats_analyse(self):
        """
        Affiche les resultats dans les zones de texte
        apres que l analyse est terminee
        """
        if not self.resultat_courant:
            return

        resume_medecin = self.resultat_courant.get("resume_medecin", "Non disponible")
        resume_patient = self.resultat_courant.get("resume_patient", "Non disponible")
        urgence        = self.resultat_courant.get("niveau_urgence", "normal")

        self.zone_medecin.delete("0.0", "end")
        self.zone_medecin.insert("0.0", resume_medecin)

        self.zone_patient.delete("0.0", "end")
        self.zone_patient.insert("0.0", resume_patient)

        couleur_urgence = {
            "normal"   : COULEUR_SUCCES,
            "moderate" : COULEUR_WARNING,
            "urgent"   : COULEUR_URGENCE
        }.get(urgence, COULEUR_SUCCES)

        duree  = self.resultat_courant.get("duree_analyse", 0)
        rougeL = self.resultat_courant.get("rougeL", 0)

        self.label_progression.configure(
            text        = f"Analyse terminee en {duree}s  |  Urgence : {urgence.upper()}  |  ROUGE-L : {rougeL:.3f}",
            text_color  = couleur_urgence
        )


    def _exporter_pdf_medecin(self):
        """
        Exporte le rapport PDF pour le medecin
        et l ouvre automatiquement
        """
        if not self.resultat_courant:
            messagebox.showwarning("Attention", "Aucune analyse disponible.")
            return

        chemin = filedialog.asksaveasfilename(
            defaultextension = ".pdf",
            filetypes        = [("PDF", "*.pdf")],
            initialfile      = "rapport_medecin.pdf",
            title            = "Sauvegarder le rapport medecin"
        )

        if chemin:
            generer_pdf_medecin(self.resultat_courant, chemin)
            self._ouvrir_fichier(chemin)
            messagebox.showinfo("Succes", f"Rapport medecin sauvegarde !")


    def _exporter_pdf_patient(self):
        """
        Exporte le rapport PDF pour le patient
        et l ouvre automatiquement
        """
        if not self.resultat_courant:
            messagebox.showwarning("Attention", "Aucune analyse disponible.")
            return

        chemin = filedialog.asksaveasfilename(
            defaultextension = ".pdf",
            filetypes        = [("PDF", "*.pdf")],
            initialfile      = "rapport_patient.pdf",
            title            = "Sauvegarder le rapport patient"
        )

        if chemin:
            generer_pdf_patient(self.resultat_courant, chemin)
            self._ouvrir_fichier(chemin)
            messagebox.showinfo("Succes", f"Rapport patient sauvegarde !")


    def _exporter_tous_pdf(self):
        """
        Exporte les deux PDF en une seule operation
        """
        if not self.resultat_courant:
            messagebox.showwarning("Attention", "Aucune analyse disponible.")
            return

        dossier = filedialog.askdirectory(title="Choisir le dossier de sauvegarde")

        if dossier:
            pdfs = generer_tous_les_pdf(self.resultat_courant, dossier)
            self._ouvrir_fichier(pdfs["pdf_medecin"])
            messagebox.showinfo(
                "Succes",
                f"Les deux PDF ont ete generes !\n\n"
                f"Medecin : {os.path.basename(pdfs['pdf_medecin'])}\n"
                f"Patient : {os.path.basename(pdfs['pdf_patient'])}"
            )


    def _ouvrir_fichier(self, chemin: str):
        """
        Ouvre un fichier avec l application par defaut du systeme
        """
        try:
            os.startfile(chemin)
        except Exception:
            pass


    # --------------------------------------------------------
    # Page Historique
    # --------------------------------------------------------

    def _afficher_historique(self):
        """
        Page historique : tableau de toutes les analyses passees
        avec possibilite de revoir les resultats
        """
        self._naviguer_silencieux("historique")

        frame = ctk.CTkFrame(
            self.zone_contenu,
            fg_color = COULEUR_FOND
        )
        frame.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)
        frame.grid_columnconfigure(0, weight=1)
        frame.grid_rowconfigure(2, weight=1)

        ctk.CTkLabel(
            frame,
            text        = "Historique des analyses",
            font        = POLICE_TITRE,
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, sticky="w", pady=(0,5))

        ctk.CTkLabel(
            frame,
            text        = "Toutes vos analyses precedentes",
            font        = POLICE_PETITE,
            text_color  = COULEUR_TEXTE2
        ).grid(row=1, column=0, sticky="w", pady=(0,15))

        # Tableau scrollable
        scroll = ctk.CTkScrollableFrame(
            frame,
            fg_color = COULEUR_CARTE,
            corner_radius = 12
        )
        scroll.grid(row=2, column=0, sticky="nsew", pady=5)

        for col in range(6):
            scroll.grid_columnconfigure(col, weight=1)

        # Entetes
        entetes = ["#", "Date", "Fichier", "Urgence", "ROUGE-L", "Duree"]
        couleurs_entetes = [COULEUR_ACCENT2] * 6

        for col, (entete, couleur) in enumerate(zip(entetes, couleurs_entetes)):
            ctk.CTkLabel(
                scroll,
                text        = entete,
                font        = ("Segoe UI", 12, "bold"),
                text_color  = couleur
            ).grid(row=0, column=col, padx=15, pady=12, sticky="w")

        # Separateur
        ctk.CTkFrame(
            scroll,
            fg_color = COULEUR_ACCENT,
            height   = 1
        ).grid(row=1, column=0, columnspan=6, sticky="ew", padx=10, pady=2)

        # Donnees historique
        historique = charger_historique()

        if len(historique) == 0:
            ctk.CTkLabel(
                scroll,
                text        = "Aucune analyse dans l historique.\nLancez votre premiere analyse !",
                font        = POLICE_NORMALE,
                text_color  = COULEUR_TEXTE2,
                justify     = "center"
            ).grid(row=2, column=0, columnspan=6, pady=40)
            return

        for ligne, (_, row) in enumerate(historique.iterrows()):
            bg = COULEUR_CARTE if ligne % 2 == 0 else COULEUR_CARTE2

            urgence = str(row.get("niveau_urgence", "normal"))
            urgence_couleur = {
                "normal"   : COULEUR_SUCCES,
                "moderate" : COULEUR_WARNING,
                "urgent"   : COULEUR_URGENCE
            }.get(urgence, COULEUR_SUCCES)

            valeurs = [
                str(row.get("id", "")),
                str(row.get("date_analyse", ""))[:16],
                str(row.get("nom_fichier", ""))[:25],
                urgence.upper(),
                f"{float(row.get('rougeL', 0)):.3f}",
                f"{float(row.get('duree_analyse', 0)):.1f}s"
            ]

            couleurs_vals = [
                COULEUR_TEXTE2,
                COULEUR_TEXTE,
                COULEUR_TEXTE,
                urgence_couleur,
                COULEUR_SUCCES,
                COULEUR_TEXTE2
            ]

            for col, (valeur, couleur) in enumerate(zip(valeurs, couleurs_vals)):
                ctk.CTkLabel(
                    scroll,
                    text        = valeur,
                    font        = ("Segoe UI", 11),
                    text_color  = couleur
                ).grid(row=ligne+2, column=col, padx=15, pady=8, sticky="w")


    # --------------------------------------------------------
    # Page Evaluation
    # --------------------------------------------------------

    def _afficher_evaluation(self):
        """
        Page evaluation : resultats ROUGE et ablation study
        avec graphiques comparatifs
        """
        self._naviguer_silencieux("evaluation")

        scroll = ctk.CTkScrollableFrame(
            self.zone_contenu,
            fg_color = COULEUR_FOND
        )
        scroll.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)
        scroll.grid_columnconfigure(0, weight=1)

        ctk.CTkLabel(
            scroll,
            text        = "Evaluation et Ablation Study",
            font        = POLICE_TITRE,
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, sticky="w", pady=(0,5))

        ctk.CTkLabel(
            scroll,
            text        = "Comparaison systeme multi-agents vs monolithique",
            font        = POLICE_PETITE,
            text_color  = COULEUR_TEXTE2
        ).grid(row=1, column=0, sticky="w", pady=(0,20))

        # Charger les donnees ablation
        try:
            connexion   = sqlite3.connect(DB_PATH)
            df_ablation = pd.read_sql_query(
                "SELECT * FROM ablation ORDER BY date DESC",
                connexion
            )
            connexion.close()
        except Exception:
            df_ablation = pd.DataFrame()

        if len(df_ablation) == 0:
            frame_vide = ctk.CTkFrame(
                scroll,
                fg_color        = COULEUR_CARTE,
                corner_radius   = 12
            )
            frame_vide.grid(row=2, column=0, sticky="ew", pady=10)

            ctk.CTkLabel(
                frame_vide,
                text        = "Aucune donnee d evaluation disponible.\nLancez la cellule 8 pour generer les resultats.",
                font        = POLICE_NORMALE,
                text_color  = COULEUR_TEXTE2,
                justify     = "center"
            ).pack(pady=40)

            ctk.CTkButton(
                frame_vide,
                text            = "Lancer l evaluation maintenant",
                font            = ("Segoe UI", 12, "bold"),
                fg_color        = COULEUR_ACCENT,
                hover_color     = COULEUR_BTN_HOVER,
                height          = 40,
                corner_radius   = 8,
                command         = self._lancer_evaluation_rapide
            ).pack(pady=(0,20))
            return

        # Graphique ablation study
        self._afficher_graphique_ablation(scroll, df_ablation)

        # Tableau comparatif
        self._afficher_tableau_ablation(scroll, df_ablation)


    def _afficher_graphique_ablation(self, parent, df_ablation):
        """
        Affiche le graphique comparatif multi-agents vs monolithique
        """
        frame = ctk.CTkFrame(
            parent,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12
        )
        frame.grid(row=2, column=0, sticky="ew", pady=10)

        ctk.CTkLabel(
            frame,
            text        = "Comparaison ROUGE : Multi-agents vs Monolithique",
            font        = ("Segoe UI", 14, "bold"),
            text_color  = COULEUR_TEXTE
        ).pack(padx=20, pady=(15,10), anchor="w")

        metriques = ["rouge1", "rouge2", "rougeL"]
        noms      = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]

        moyennes_mono  = [df_ablation[f"{m}_mono"].mean()  for m in metriques]
        moyennes_multi = [df_ablation[f"{m}_multi"].mean() for m in metriques]

        fig, ax = plt.subplots(figsize=(10, 4))
        fig.patch.set_facecolor("#1a2535")
        ax.set_facecolor("#0f1923")

        x      = range(len(noms))
        largeur = 0.35

        barres_mono  = ax.bar(
            [i - largeur/2 for i in x],
            moyennes_mono,
            largeur,
            label       = "Monolithique",
            color       = "#e74c3c",
            edgecolor   = "none"
        )
        barres_multi = ax.bar(
            [i + largeur/2 for i in x],
            moyennes_multi,
            largeur,
            label       = "Multi-agents",
            color       = "#2ecc71",
            edgecolor   = "none"
        )

        for barre in barres_mono:
            ax.text(
                barre.get_x() + barre.get_width()/2,
                barre.get_height() + 0.005,
                f"{barre.get_height():.3f}",
                ha="center", color="white", fontsize=9
            )
        for barre in barres_multi:
            ax.text(
                barre.get_x() + barre.get_width()/2,
                barre.get_height() + 0.005,
                f"{barre.get_height():.3f}",
                ha="center", color="white", fontsize=9
            )

        ax.set_xticks(list(x))
        ax.set_xticklabels(noms, color="#bdc3c7")
        ax.tick_params(colors="#bdc3c7")
        ax.spines["bottom"].set_color("#2c3e50")
        ax.spines["left"].set_color("#2c3e50")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(facecolor="#1a2535", labelcolor="white", fontsize=10)
        ax.set_ylabel("Score ROUGE moyen", color="#bdc3c7")
        ax.grid(True, alpha=0.15, color="white", axis="y")
        ax.set_title("Ablation Study — Multi-agents vs Monolithique", color="white")

        plt.tight_layout()
        canvas = FigureCanvasTkAgg(fig, master=frame)
        canvas.draw()
        canvas.get_tk_widget().pack(padx=15, pady=(0,15), fill="both")
        plt.close(fig)


    def _afficher_tableau_ablation(self, parent, df_ablation):
        """
        Affiche le tableau comparatif des scores
        """
        frame = ctk.CTkFrame(
            parent,
            fg_color        = COULEUR_CARTE,
            corner_radius   = 12
        )
        frame.grid(row=3, column=0, sticky="ew", pady=10)
        frame.grid_columnconfigure((0,1,2,3), weight=1)

        ctk.CTkLabel(
            frame,
            text        = "Tableau comparatif des scores",
            font        = ("Segoe UI", 14, "bold"),
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, columnspan=4, padx=20, pady=(15,10), sticky="w")

        entetes = ["Metrique", "Monolithique", "Multi-Agents", "Gain"]
        for col, entete in enumerate(entetes):
            ctk.CTkLabel(
                frame,
                text        = entete,
                font        = ("Segoe UI", 12, "bold"),
                text_color  = COULEUR_ACCENT2
            ).grid(row=1, column=col, padx=20, pady=8)

        metriques = [
            ("ROUGE-1", "rouge1"),
            ("ROUGE-2", "rouge2"),
            ("ROUGE-L", "rougeL")
        ]

        for ligne, (nom, cle) in enumerate(metriques):
            mono  = df_ablation[f"{cle}_mono"].mean()
            multi = df_ablation[f"{cle}_multi"].mean()
            gain  = multi - mono
            signe = "+" if gain >= 0 else ""
            couleur_gain = COULEUR_SUCCES if gain >= 0 else COULEUR_URGENCE

            valeurs = [nom, f"{mono:.4f}", f"{multi:.4f}", f"{signe}{gain:.4f}"]
            couleurs_vals = [COULEUR_TEXTE, COULEUR_TEXTE2, COULEUR_TEXTE, couleur_gain]

            for col, (valeur, couleur) in enumerate(zip(valeurs, couleurs_vals)):
                ctk.CTkLabel(
                    frame,
                    text        = valeur,
                    font        = ("Segoe UI", 12),
                    text_color  = couleur
                ).grid(row=ligne+2, column=col, padx=20, pady=8)

        ctk.CTkFrame(
            frame,
            fg_color = COULEUR_BORDURE,
            height   = 1
        ).grid(row=5, column=0, columnspan=4, sticky="ew", padx=15, pady=5)

        rougeL_mono  = df_ablation["rougeL_mono"].mean()
        rougeL_multi = df_ablation["rougeL_multi"].mean()

        if rougeL_multi > rougeL_mono and rougeL_mono > 0:
            gain_pct    = round((rougeL_multi - rougeL_mono) / rougeL_mono * 100, 1)
            conclusion  = f"Le systeme multi-agents surpasse le monolithique de {gain_pct}% sur ROUGE-L"
            couleur_c   = COULEUR_SUCCES
        else:
            conclusion  = "Les deux systemes ont des performances comparables"
            couleur_c   = COULEUR_WARNING

        ctk.CTkLabel(
            frame,
            text        = conclusion,
            font        = ("Segoe UI", 12, "bold"),
            text_color  = couleur_c
        ).grid(row=6, column=0, columnspan=4, padx=20, pady=(5,20))


    def _lancer_evaluation_rapide(self):
        """
        Lance une evaluation rapide sur 3 rapports depuis l interface
        """
        def evaluer():
            try:
                evaluer_systeme(nombre_rapports=3)
                self.after(100, lambda: messagebox.showinfo(
                    "Succes",
                    "Evaluation terminee ! Rechargez la page."
                ))
                self.after(200, self._afficher_evaluation)
            except Exception as e:
                self.after(100, lambda: messagebox.showerror("Erreur", str(e)))

        thread = threading.Thread(target=evaluer)
        thread.daemon = True
        thread.start()
        messagebox.showinfo("Info", "Evaluation lancee en arriere plan (10-15 min).")


    # --------------------------------------------------------
    # Page A propos
    # --------------------------------------------------------

    def _afficher_apropos(self):
        """
        Page a propos : description du projet et de l equipe
        """
        self._naviguer_silencieux("apropos")

        scroll = ctk.CTkScrollableFrame(
            self.zone_contenu,
            fg_color = COULEUR_FOND
        )
        scroll.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)
        scroll.grid_columnconfigure(0, weight=1)

        ctk.CTkLabel(
            scroll,
            text        = "A propos de RadioAI Pro",
            font        = POLICE_TITRE,
            text_color  = COULEUR_TEXTE
        ).grid(row=0, column=0, sticky="w", pady=(0,20))

        infos = [
            ("Projet",          "Structuration et synthese agentique de comptes rendus de radiologie"),
            ("Ecole",           "I3AFD — Intelligence Artificielle et Fouille de Donnees"),
            ("Lieu",            "Yaounde, Cameroun — Mars-Avril 2026"),
            ("Dataset",         "OpenI / IU X-Ray — 3955 rapports radiologiques"),
            ("Modele IA",       "Mistral 7B via Ollama (execution locale CPU)"),
            ("Framework",       "LangGraph — Orchestration multi-agents"),
            ("Evaluation",      "ROUGE-1 / ROUGE-2 / ROUGE-L + Ablation Study"),
            ("Interface",       "CustomTkinter — Application desktop Windows"),
        ]

        for ligne, (cle, valeur) in enumerate(infos):
            frame_info = ctk.CTkFrame(
                scroll,
                fg_color        = COULEUR_CARTE,
                corner_radius   = 8
            )
            frame_info.grid(row=ligne+1, column=0, sticky="ew", pady=4)
            frame_info.grid_columnconfigure(1, weight=1)

            ctk.CTkLabel(
                frame_info,
                text        = cle,
                font        = ("Segoe UI", 12, "bold"),
                text_color  = COULEUR_ACCENT2,
                width       = 140,
                anchor      = "w"
            ).grid(row=0, column=0, padx=15, pady=12, sticky="w")

            ctk.CTkLabel(
                frame_info,
                text        = valeur,
                font        = ("Segoe UI", 12),
                text_color  = COULEUR_TEXTE,
                anchor      = "w"
            ).grid(row=0, column=1, padx=15, pady=12, sticky="w")


    # --------------------------------------------------------
    # Methode utilitaire : navigation sans boucle
    # --------------------------------------------------------

    def _naviguer_silencieux(self, page: str):
        """
        Met a jour uniquement le style des boutons
        sans relancer la page (evite la recursion)
        """
        for cle, btn in self.boutons_nav.items():
            btn.configure(
                fg_color   = "transparent",
                text_color = COULEUR_TEXTE2
            )
        if page in self.boutons_nav:
            self.boutons_nav[page].configure(
                fg_color   = COULEUR_ACCENT,
                text_color = "white"
            )


# ============================================================
# Lancement de l application
# ============================================================

print("Demarrage de RadioAI Pro...")
print("L application va s ouvrir dans une nouvelle fenetre.")


Demarrage de RadioAI Pro...
L application va s ouvrir dans une nouvelle fenetre.


In [ ]:
app = RadioAIPro()
app.mainloop()

Extraction depuis Image : ARCHITECTURE-RNER-VF-1536x949.png


In [18]:
pwd()

'C:\\Users\\TOE'